In [1]:
# ============================================================
# NOTEBOOK 5 — CONTINENTAL AND COUNTRY ANALYTICAL EVIDENCE
#
# STEP 1:
# Inventory the validated datasets and model outputs before
# constructing 2023 predictor rows or producing 2024 risk scores.
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

try:
    import pyarrow.parquet as pq
except ImportError as error:
    raise ImportError(
        "PyArrow is required to inspect the Parquet files."
    ) from error


# ------------------------------------------------------------
# 1. Define project locations
# ------------------------------------------------------------

project_directory = Path(
    "/Users/adewale/Documents/food_security_predictor"
)

processed_data_directory = (
    project_directory
    / "data"
    / "processed"
    / "africa_first"
)

model_output_directory = (
    project_directory
    / "models"
    / "africa_first"
)

assert processed_data_directory.exists(), (
    f"Processed-data directory not found: "
    f"{processed_data_directory}"
)

assert model_output_directory.exists(), (
    f"Model-output directory not found: "
    f"{model_output_directory}"
)


# ------------------------------------------------------------
# 2. Find saved analytical outputs
# ------------------------------------------------------------

relevant_extensions = {
    ".parquet",
    ".csv",
    ".json",
    ".pkl",
    ".pickle",
}

candidate_files = sorted(
    [
        file_path
        for directory in [
            processed_data_directory,
            model_output_directory,
        ]
        for file_path in directory.rglob("*")
        if (
            file_path.is_file()
            and file_path.suffix.lower()
            in relevant_extensions
        )
    ],
    key=lambda path: str(path).lower(),
)

assert candidate_files, (
    "No relevant saved outputs were found."
)


# ------------------------------------------------------------
# 3. Safely inspect each file
# ------------------------------------------------------------

inventory_records = []

important_columns = [
    "Area",
    "Item Code",
    "Item",
    "Year",
    "target_year",
    "shortage_next_year",
    "Temporal split",
]

for file_path in candidate_files:

    record = {
        "Directory":
            (
                "Processed data"
                if processed_data_directory
                in file_path.parents
                else "Model outputs"
            ),
        "File": file_path.name,
        "Type": file_path.suffix.lower(),
        "Size MB":
            file_path.stat().st_size / (1024 ** 2),
        "Rows": np.nan,
        "Columns": np.nan,
        "Year start": np.nan,
        "Year end": np.nan,
        "Target year start": np.nan,
        "Target year end": np.nan,
        "Has shortage target": False,
        "Has temporal split": False,
        "Inspection status": "Not inspected",
    }

    try:

        if file_path.suffix.lower() == ".parquet":

            parquet_file = pq.ParquetFile(file_path)

            available_columns = (
                parquet_file.schema_arrow.names
            )

            record["Rows"] = (
                parquet_file.metadata.num_rows
            )

            record["Columns"] = len(
                available_columns
            )

            record["Has shortage target"] = (
                "shortage_next_year"
                in available_columns
            )

            record["Has temporal split"] = (
                "Temporal split"
                in available_columns
            )

            columns_to_read = [
                column
                for column in [
                    "Year",
                    "target_year",
                ]
                if column in available_columns
            ]

            if columns_to_read:

                date_check = pd.read_parquet(
                    file_path,
                    columns=columns_to_read,
                )

                if "Year" in date_check.columns:
                    record["Year start"] = (
                        date_check["Year"].min()
                    )
                    record["Year end"] = (
                        date_check["Year"].max()
                    )

                if "target_year" in date_check.columns:
                    record["Target year start"] = (
                        date_check[
                            "target_year"
                        ].min()
                    )
                    record["Target year end"] = (
                        date_check[
                            "target_year"
                        ].max()
                    )

            record["Inspection status"] = (
                "Parquet metadata inspected"
            )

        elif file_path.suffix.lower() == ".csv":

            csv_preview = pd.read_csv(
                file_path,
                nrows=5,
            )

            record["Columns"] = len(
                csv_preview.columns
            )

            record["Has shortage target"] = (
                "shortage_next_year"
                in csv_preview.columns
            )

            record["Has temporal split"] = (
                "Temporal split"
                in csv_preview.columns
            )

            record["Inspection status"] = (
                "CSV structure inspected"
            )

        elif file_path.suffix.lower() == ".json":

            with open(
                file_path,
                "r",
                encoding="utf-8",
            ) as json_file:
                json.load(json_file)

            record["Inspection status"] = (
                "Valid JSON"
            )

        else:
            # Pickled models are deliberately not loaded here.
            record["Inspection status"] = (
                "Model file identified; not loaded"
            )

    except Exception as inspection_error:

        record["Inspection status"] = (
            f"Inspection failed: "
            f"{type(inspection_error).__name__}: "
            f"{inspection_error}"
        )

    inventory_records.append(record)


output_inventory = pd.DataFrame(
    inventory_records
)


# ------------------------------------------------------------
# 4. Display the complete inventory
# ------------------------------------------------------------

print("Saved Africa-first analytical outputs:")

display(
    output_inventory.style.format(
        {
            "Size MB": "{:.4f}",
            "Rows": "{:,.0f}",
            "Columns": "{:,.0f}",
            "Year start": "{:.0f}",
            "Year end": "{:.0f}",
            "Target year start": "{:.0f}",
            "Target year end": "{:.0f}",
        },
        na_rep="—",
    )
)


# ------------------------------------------------------------
# 5. Isolate likely feature and analytical datasets
# ------------------------------------------------------------

dataset_inventory = output_inventory.loc[
    output_inventory["Type"].isin(
        [".parquet", ".csv"]
    )
].copy()

likely_feature_files = dataset_inventory.loc[
    dataset_inventory["File"].str.contains(
        (
            "feature|model_ready|analytical|panel|"
            "shortage|supervised"
        ),
        case=False,
        regex=True,
        na=False,
    )
].copy()

print("\nLikely analytical or feature datasets:")

display(
    likely_feature_files.style.format(
        {
            "Size MB": "{:.4f}",
            "Rows": "{:,.0f}",
            "Columns": "{:,.0f}",
            "Year start": "{:.0f}",
            "Year end": "{:.0f}",
            "Target year start": "{:.0f}",
            "Target year end": "{:.0f}",
        },
        na_rep="—",
    )
)


# ------------------------------------------------------------
# 6. Check whether any table already contains 2023 predictors
# ------------------------------------------------------------

files_reaching_2023 = dataset_inventory.loc[
    pd.to_numeric(
        dataset_inventory["Year end"],
        errors="coerce",
    ).ge(2023)
].copy()

print("\nDatasets containing predictor year 2023:")

if files_reaching_2023.empty:
    print(
        "No inspected Parquet dataset currently reaches "
        "predictor year 2023."
    )
else:
    display(
        files_reaching_2023.style.format(
            {
                "Size MB": "{:.4f}",
                "Rows": "{:,.0f}",
                "Columns": "{:,.0f}",
                "Year start": "{:.0f}",
                "Year end": "{:.0f}",
                "Target year start": "{:.0f}",
                "Target year end": "{:.0f}",
            },
            na_rep="—",
        )
    )


# ------------------------------------------------------------
# 7. Confirm the operational model files are present
# ------------------------------------------------------------

required_model_files = [
    "africa_shortage_operational_model_2010_2022.pkl",
    "shortage_operational_feature_schema.csv",
    "shortage_operational_model_manifest.json",
]

model_file_check = pd.DataFrame(
    {
        "Required model output":
            required_model_files,
        "Exists": [
            (
                model_output_directory
                / filename
            ).exists()
            for filename in required_model_files
        ],
    }
)

print("\nOperational-model file check:")
display(model_file_check)

assert model_file_check["Exists"].all(), (
    "One or more required operational-model files "
    "are missing."
)

print(
    "\nInventory completed successfully.\n"
    "No data or model file was modified."
)

Saved Africa-first analytical outputs:


,Directory,File,Type,Size MB,Rows,Columns,Year start,Year end,Target year start,Target year end,Has shortage target,Has temporal split,Inspection status
0,Processed data,africa_analytical_panel_2010_2023.parquet,.parquet,0.3018,"4,816",84,2010,2023,—,—,False,False,Parquet metadata inspected
1,Processed data,africa_country_scope.csv,.csv,0.0038,—,10,—,—,—,—,False,False,CSV structure inspected
2,Processed data,africa_model_ready_shortage_features_2010_2022.parquet,.parquet,1.1667,"3,248",138,2010,2022,2011,2023,True,True,Parquet metadata inspected
3,Processed data,africa_population_2010_2023.parquet,.parquet,0.0157,602,9,2010,2023,—,—,False,False,Parquet metadata inspected
4,Processed data,africa_selected_commodities_long_2010_2023.parquet,.parquet,0.2377,"77,994",13,2010,2023,—,—,False,False,Parquet metadata inspected
5,Processed data,africa_supervised_shortage_panel_2010_2022.parquet,.parquet,0.2629,"3,248",87,2010,2022,2011,2023,True,True,Parquet metadata inspected
6,Processed data,candidate_selection_evidence.csv,.csv,0.0023,—,17,—,—,—,—,False,False,CSV structure inspected
7,Processed data,selected_commodity_metadata.csv,.csv,0.0010,—,5,—,—,—,—,False,False,CSV structure inspected
8,Processed data,shortage_feature_registry.csv,.csv,0.0153,—,7,—,—,—,—,False,False,CSV structure inspected
9,Processed data,shortage_feature_specification.json,.json,0.0110,—,—,—,—,—,—,False,False,Valid JSON



Likely analytical or feature datasets:


,Directory,File,Type,Size MB,Rows,Columns,Year start,Year end,Target year start,Target year end,Has shortage target,Has temporal split,Inspection status
0,Processed data,africa_analytical_panel_2010_2023.parquet,.parquet,0.3018,"4,816",84,2010,2023,—,—,False,False,Parquet metadata inspected
2,Processed data,africa_model_ready_shortage_features_2010_2022.parquet,.parquet,1.1667,"3,248",138,2010,2022,2011,2023,True,True,Parquet metadata inspected
5,Processed data,africa_supervised_shortage_panel_2010_2022.parquet,.parquet,0.2629,"3,248",87,2010,2022,2011,2023,True,True,Parquet metadata inspected
8,Processed data,shortage_feature_registry.csv,.csv,0.0153,—,7,—,—,—,—,False,False,CSV structure inspected
10,Processed data,shortage_modelling_split_summary.csv,.csv,0.0004,—,12,—,—,—,—,False,True,CSV structure inspected
11,Processed data,shortage_preprocessing_policy.csv,.csv,0.0007,—,3,—,—,—,—,False,False,CSV structure inspected
12,Processed data,shortage_target_by_commodity.csv,.csv,0.0004,—,6,—,—,—,—,False,False,CSV structure inspected
13,Processed data,shortage_target_by_year.csv,.csv,0.0004,—,5,—,—,—,—,False,False,CSV structure inspected
14,Processed data,shortage_target_definition.csv,.csv,0.0007,—,15,—,—,—,—,False,False,CSV structure inspected
15,Processed data,shortage_temporal_split_summary.csv,.csv,0.0004,—,13,—,—,—,—,False,True,CSV structure inspected



Datasets containing predictor year 2023:


,Directory,File,Type,Size MB,Rows,Columns,Year start,Year end,Target year start,Target year end,Has shortage target,Has temporal split,Inspection status
0,Processed data,africa_analytical_panel_2010_2023.parquet,.parquet,0.3018,"4,816",84,2010,2023,—,—,False,False,Parquet metadata inspected
3,Processed data,africa_population_2010_2023.parquet,.parquet,0.0157,602,9,2010,2023,—,—,False,False,Parquet metadata inspected
4,Processed data,africa_selected_commodities_long_2010_2023.parquet,.parquet,0.2377,"77,994",13,2010,2023,—,—,False,False,Parquet metadata inspected



Operational-model file check:


,Required model output,Exists
0,africa_shortage_operational_model_2010_2022.pkl,True
1,shortage_operational_feature_schema.csv,True
2,shortage_operational_model_manifest.json,True



Inventory completed successfully.
No data or model file was modified.


In [2]:
# ============================================================
# NOTEBOOK 5 — STEP 2
# INSPECT THE LOCKED FEATURE-ENGINEERING SPECIFICATION
#
# This cell reads and validates the saved Notebook 3 evidence.
# It does not create features or generate predictions.
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Define input files
# ------------------------------------------------------------

analytical_panel_path = (
    processed_data_directory
    / "africa_analytical_panel_2010_2023.parquet"
)

model_ready_path = (
    processed_data_directory
    / "africa_model_ready_shortage_features_2010_2022.parquet"
)

feature_registry_path = (
    processed_data_directory
    / "shortage_feature_registry.csv"
)

feature_specification_path = (
    processed_data_directory
    / "shortage_feature_specification.json"
)

operational_schema_path = (
    model_output_directory
    / "shortage_operational_feature_schema.csv"
)

required_paths = [
    analytical_panel_path,
    model_ready_path,
    feature_registry_path,
    feature_specification_path,
    operational_schema_path,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

assert not missing_paths, (
    "Required Notebook 3 evidence is missing:\n"
    + "\n".join(missing_paths)
)


# ------------------------------------------------------------
# 2. Load the saved evidence
# ------------------------------------------------------------

analytical_panel = pd.read_parquet(
    analytical_panel_path
)

model_ready_history = pd.read_parquet(
    model_ready_path
)

feature_registry = pd.read_csv(
    feature_registry_path
)

operational_feature_schema = pd.read_csv(
    operational_schema_path
)

with open(
    feature_specification_path,
    "r",
    encoding="utf-8",
) as specification_file:
    feature_specification = json.load(
        specification_file
    )


# ------------------------------------------------------------
# 3. Validate the principal dimensions
# ------------------------------------------------------------

assert len(analytical_panel) == 4816
assert analytical_panel["Year"].min() == 2010
assert analytical_panel["Year"].max() == 2023

assert len(model_ready_history) == 3248
assert model_ready_history["Year"].min() == 2010
assert model_ready_history["Year"].max() == 2022

assert len(operational_feature_schema) == 129

assert not analytical_panel.duplicated(
    ["Area", "Item Code", "Year"]
).any()

assert not model_ready_history.duplicated(
    ["Area", "Item Code", "Year"]
).any()


# ------------------------------------------------------------
# 4. Check the 2023 analytical rows
# ------------------------------------------------------------

analytical_2023 = analytical_panel.loc[
    analytical_panel["Year"].eq(2023)
].copy()

expected_country_commodity_grid = (
    analytical_panel["Area"].nunique()
    * analytical_panel["Item Code"].nunique()
)

rows_2023_summary = pd.DataFrame(
    {
        "Measure": [
            "2023 analytical rows",
            "Expected full country-commodity grid",
            "Countries represented",
            "Commodities represented",
            "Unique country-commodity pairs",
            "Duplicate country-commodity-year keys",
        ],
        "Value": [
            len(analytical_2023),
            expected_country_commodity_grid,
            analytical_2023["Area"].nunique(),
            analytical_2023["Item Code"].nunique(),
            analytical_2023[
                ["Area", "Item Code"]
            ].drop_duplicates().shape[0],
            analytical_2023.duplicated(
                ["Area", "Item Code", "Year"]
            ).sum(),
        ],
    }
)

print("Availability of 2023 analytical observations:")
display(rows_2023_summary)


# ------------------------------------------------------------
# 5. Display the analytical-panel structure
# ------------------------------------------------------------

analytical_structure = pd.DataFrame(
    {
        "Column order":
            np.arange(
                1,
                len(analytical_panel.columns) + 1,
            ),
        "Analytical-panel column":
            analytical_panel.columns,
        "Data type": [
            str(analytical_panel[column].dtype)
            for column in analytical_panel.columns
        ],
        "Missing in 2023": [
            int(analytical_2023[column].isna().sum())
            for column in analytical_panel.columns
        ],
        "Non-missing in 2023": [
            int(
                analytical_2023[column]
                .notna()
                .sum()
            )
            for column in analytical_panel.columns
        ],
    }
)

print("\nComplete analytical-panel structure:")
display(analytical_structure)


# ------------------------------------------------------------
# 6. Identify metadata and predictor columns in the saved
#    model-ready history
# ------------------------------------------------------------

schema_feature_column_candidates = [
    column
    for column in operational_feature_schema.columns
    if column.strip().lower() == "feature"
]

assert len(schema_feature_column_candidates) == 1, (
    "Could not uniquely identify the Feature column "
    "in the operational schema."
)

schema_feature_column = (
    schema_feature_column_candidates[0]
)

locked_feature_names = (
    operational_feature_schema[
        schema_feature_column
    ]
    .astype(str)
    .tolist()
)

assert len(locked_feature_names) == 129
assert len(set(locked_feature_names)) == 129

missing_locked_features_in_history = sorted(
    set(locked_feature_names)
    - set(model_ready_history.columns)
)

assert not missing_locked_features_in_history, (
    "The historical model-ready table does not contain "
    "all locked features:\n"
    f"{missing_locked_features_in_history}"
)

non_feature_columns = [
    column
    for column in model_ready_history.columns
    if column not in locked_feature_names
]

model_ready_structure_summary = pd.DataFrame(
    {
        "Measure": [
            "Model-ready columns",
            "Locked raw predictors",
            "Non-feature columns",
            "Historical rows",
            "Historical predictor-year start",
            "Historical predictor-year end",
        ],
        "Value": [
            len(model_ready_history.columns),
            len(locked_feature_names),
            len(non_feature_columns),
            len(model_ready_history),
            model_ready_history["Year"].min(),
            model_ready_history["Year"].max(),
        ],
    }
)

print("\nHistorical model-ready structure:")
display(model_ready_structure_summary)

print("\nNon-feature columns retained in the model-ready table:")
display(
    pd.DataFrame(
        {
            "Column": non_feature_columns,
            "Data type": [
                str(
                    model_ready_history[
                        column
                    ].dtype
                )
                for column in non_feature_columns
            ],
        }
    )
)


# ------------------------------------------------------------
# 7. Display the exact 129-feature input schema
# ------------------------------------------------------------

print("\nLocked operational feature schema:")
display(operational_feature_schema)


# ------------------------------------------------------------
# 8. Display the saved Notebook 3 feature registry
# ------------------------------------------------------------

print("\nNotebook 3 feature registry:")
display(feature_registry)

print(
    "\nFeature-registry columns:",
    feature_registry.columns.tolist(),
)


# ------------------------------------------------------------
# 9. Display the complete saved feature specification
# ------------------------------------------------------------

print("\nFeature-specification top-level keys:")
print(list(feature_specification.keys()))

print("\nComplete saved feature specification:")
print(
    json.dumps(
        feature_specification,
        indent=2,
        ensure_ascii=False,
    )
)


# ------------------------------------------------------------
# 10. Compare the analytical panel with the locked features
# ------------------------------------------------------------

directly_available_features = [
    feature
    for feature in locked_feature_names
    if feature in analytical_panel.columns
]

engineered_features = [
    feature
    for feature in locked_feature_names
    if feature not in analytical_panel.columns
]

feature_origin_summary = pd.DataFrame(
    {
        "Feature origin": [
            "Already present in analytical panel",
            "Must be engineered from analytical history",
            "Total locked features",
        ],
        "Number of features": [
            len(directly_available_features),
            len(engineered_features),
            len(locked_feature_names),
        ],
    }
)

print("\nLocked-feature origin summary:")
display(feature_origin_summary)

print(
    "\nLocked features already present "
    "in the analytical panel:"
)
print(directly_available_features)

print(
    "\nLocked features requiring reconstruction:"
)
print(engineered_features)


# ------------------------------------------------------------
# 11. Final specification-integrity checks
# ------------------------------------------------------------

assert len(directly_available_features) + len(
    engineered_features
) == 129

assert set(locked_feature_names).issubset(
    model_ready_history.columns
)

print(
    "\nLocked feature specification inspected successfully.\n"
    "No 2023 feature was created and no prediction was made."
)

Availability of 2023 analytical observations:


,Measure,Value
0,2023 analytical rows,344
1,Expected full country-commodity grid,344
2,Countries represented,43
3,Commodities represented,8
4,Unique country-commodity pairs,344
5,Duplicate country-commodity-year keys,0



Complete analytical-panel structure:


,Column order,Analytical-panel column,Data type,Missing in 2023,Non-missing in 2023
0,1,Area Code,int64,0,344
1,2,Area Code (M49),string,0,344
2,3,M49 Code,string,0,344
3,4,Area,str,0,344
4,5,Item Code,int64,0,344
...,...,...,...,...,...
79,80,recorded_negative_element_count,int8,0,344
80,81,supply_outcome_recorded_count,int8,0,344
81,82,supply_outcomes_complete,bool,0,344
82,83,production_status,str,0,344



Historical model-ready structure:


...


Notebook 3 feature registry:


,Feature,Feature type,Training missing values,Training missing %,Training unique values including missing,Status,Reason
0,Area,Categorical,0,0.000000,43,Retained,Contains usable training-period variation
1,Item Code,Categorical,0,0.000000,8,Retained,Contains usable training-period variation
2,production_status,Categorical,0,0.000000,4,Retained,Contains usable training-period variation
3,domestic_supply_1000t_source_present,Binary indicator,0,0.000000,1,Removed,Contains no variation in the training period
4,export_quantity_1000t_source_present,Binary indicator,0,0.000000,2,Retained,Contains usable training-period variation
...,...,...,...,...,...,...,...
139,stock_variation_share_of_domestic_supply_pct,Numeric,119,5.288889,1210,Retained,Contains usable training-period variation
140,production_kg_per_person,Numeric,263,11.688889,1862,Retained,Contains usable training-period variation
141,imports_kg_per_person,Numeric,139,6.177778,1552,Retained,Contains usable training-period variation
142,exports_kg_per_person,Numeric,476,21.155556,833,Retained,Contains usable training-period variation



Feature-registry columns: ['Feature', 'Feature type', 'Training missing values', 'Training missing %', 'Training unique values including missing', 'Status', 'Reason']

Feature-specification top-level keys:
['dataset_name', 'prediction_horizon', 'target_column', 'target_definition', 'panel_key_columns', 'observation_identifier', 'final_feature_count', 'final_feature_columns', 'categorical_features', 'binary_indicator_features', 'numeric_features', 'historical_coverage_threshold_pct', 'historical_base_features', 'sparse_current_only_features', 'removed_training_constant_features', 'manual_feature_exclusions', 'temporal_splits', 'future_information_used', 'raw_missing_values_preserved', 'raw_extreme_values_preserved']

Complete saved feature specification:
{
  "dataset_name": "Africa next-year commodity shortage modelling dataset",
  "prediction_horizon": "One year ahead",
  "target_column": "shortage_next_year",
  "target_definition": {
    "minimum_current_kcal_capita_day": 5,
    "min

,Feature origin,Number of features
0,Already present in analytical panel,38
1,Must be engineered from analytical history,91
2,Total locked features,129



Locked features already present in the analytical panel:
['Area', 'Item Code', 'production_status', 'export_quantity_1000t_source_present', 'feed_1000t_source_present', 'import_quantity_1000t_source_present', 'losses_1000t_source_present', 'other_uses_non_food_1000t_source_present', 'processing_1000t_source_present', 'production_1000t_source_present', 'seed_1000t_source_present', 'stock_variation_1000t_source_present', 'tourist_consumption_1000t_source_present', 'Year', 'Population', 'domestic_supply_1000t', 'export_quantity_1000t', 'fat_supply_g_cap_day', 'feed_1000t', 'food_1000t', 'food_supply_kcal_cap_day', 'food_supply_quantity_kg_cap_yr', 'import_quantity_1000t', 'losses_1000t', 'other_uses_non_food_1000t', 'processing_1000t', 'production_1000t', 'protein_supply_g_cap_day', 'residuals_1000t', 'seed_1000t', 'stock_variation_1000t', 'tourist_consumption_1000t', 'recorded_element_count', 'source_absent_element_count', 'source_present_missing_element_count', 'recorded_zero_element_c

In [4]:
# ============================================================
# NOTEBOOK 5 — STEP 3 CORRECTION
# COMPLETE THE HISTORICAL RECONSTRUCTION VALIDATION
#
# Area, Item Code and Year are index levels in the historical
# reference table, so retrieve them from the index when needed.
# ============================================================


# ------------------------------------------------------------
# 1. Defragment the completed table
# ------------------------------------------------------------

reconstructed_features = (
    reconstructed_features.copy()
)


# ------------------------------------------------------------
# 2. Validate all 129 reconstructed historical features
# ------------------------------------------------------------

feature_validation_records = []

categorical_features = set(
    feature_specification[
        "categorical_features"
    ]
)

historical_index_features = {
    "Area",
    "Item Code",
    "Year",
}

for feature in locked_feature_names:

    candidate = align_with_historical_reference(
        reconstructed_features[feature]
    )

    if feature in historical_index_features:

        expected = pd.Series(
            historical_reference.index
            .get_level_values(feature),
            index=historical_reference.index,
            name=feature,
        )

    else:

        assert feature in historical_reference.columns, (
            f"Historical reference does not contain "
            f"the expected feature: {feature}"
        )

        expected = historical_reference[feature]

    if feature in categorical_features:

        comparison = compare_categorical(
            candidate,
            expected,
        )

    else:

        comparison = compare_numeric(
            candidate,
            expected,
        )

    feature_validation_records.append(
        {
            "Feature": feature,
            **comparison,
        }
    )


feature_reconstruction_validation = (
    pd.DataFrame(
        feature_validation_records
    )
)

failed_feature_validation = (
    feature_reconstruction_validation.loc[
        feature_reconstruction_validation[
            "Mismatches"
        ].gt(0)
    ]
    .sort_values(
        ["Mismatches", "Feature"],
        ascending=[False, True],
    )
)

validation_summary = pd.DataFrame(
    {
        "Measure": [
            "Locked features expected",
            "Locked features reconstructed",
            "Features matching historical evidence",
            "Features with mismatches",
            "Total compared cells",
            "Total mismatched cells",
            "Reconstructed 2023 rows",
            "Infinite numeric values",
        ],
        "Value": [
            len(locked_feature_names),
            len(reconstructed_features.columns),
            int(
                feature_reconstruction_validation[
                    "Mismatches"
                ].eq(0).sum()
            ),
            len(failed_feature_validation),
            int(
                feature_reconstruction_validation[
                    "Compared values"
                ].sum()
            ),
            int(
                feature_reconstruction_validation[
                    "Mismatches"
                ].sum()
            ),
            int(
                reconstructed_features[
                    "Year"
                ].eq(2023).sum()
            ),
            int(
                np.isinf(
                    reconstructed_features
                    .select_dtypes(
                        include=[np.number]
                    )
                    .to_numpy()
                ).sum()
            ),
        ],
    }
)


# ------------------------------------------------------------
# 3. Display formula-identification evidence
# ------------------------------------------------------------

print("Percentage-change convention check:")
display(percentage_convention_check)

print("\nRolling three-year mean check:")
display(rolling_mean_check)

print(
    "\nRolling three-year standard-deviation check:"
)
display(rolling_std_check)

print("\nSupply-share convention check:")
display(share_rule_check)

print("\nPer-person conversion check:")
display(per_person_rule_check)

print("\nPopulation-log convention check:")
display(log_rule_check)


# ------------------------------------------------------------
# 4. Display the full validation result
# ------------------------------------------------------------

print(
    "\nComplete reconstruction-validation summary:"
)
display(validation_summary)

if failed_feature_validation.empty:

    print(
        "\nEvery reconstructed feature matches the saved "
        "Notebook 3 historical evidence."
    )

else:

    print(
        "\nFeatures that did not reproduce the saved "
        "historical evidence:"
    )

    display(failed_feature_validation)


# ------------------------------------------------------------
# 5. Final integrity checks
# ------------------------------------------------------------

assert failed_feature_validation.empty, (
    "The reconstructed feature system does not yet "
    "exactly reproduce Notebook 3. Do not score 2023."
)

assert len(reconstructed_features.columns) == 129

assert reconstructed_features.columns.tolist() == (
    locked_feature_names
)

assert reconstructed_features[
    "Year"
].eq(2023).sum() == 344

assert not np.isinf(
    reconstructed_features
    .select_dtypes(
        include=[np.number]
    )
    .to_numpy()
).any()

print(
    "\nFeature reconstruction passed.\n"
    "All 129 features reproduce the historical "
    "Notebook 3 evidence, and 344 provisional 2023 "
    "rows exist in memory.\n"
    "Nothing has been saved and no 2024 risk score "
    "has been generated."
)

Percentage-change convention check:


,Convention,Use absolute denominator,Values compared,Mismatches
1,Absolute previous value denominator,True,42224,0
0,Previous value denominator,False,42224,713



Rolling three-year mean check:


,Includes current year,Minimum recorded years,Values compared,Mismatches
1,True,2,45472,0
0,True,1,45472,3580
2,True,3,45472,3635
3,False,1,45472,30496
4,False,2,45472,31287
5,False,3,45472,32132



Rolling three-year standard-deviation check:


...


Population-log convention check:


,Logarithm rule,Compared values,Recorded pairs compared,Mismatches,Maximum absolute difference
1,Natural logarithm plus one,3248,3248,0,0.000000
0,Natural logarithm,3248,3248,2826,0.000011
2,Base-10 logarithm,3248,3248,3248,10.874760



Complete reconstruction-validation summary:


,Measure,Value
0,Locked features expected,129
1,Locked features reconstructed,129
2,Features matching historical evidence,129
3,Features with mismatches,0
4,Total compared cells,418992
5,Total mismatched cells,0
6,Reconstructed 2023 rows,344
7,Infinite numeric values,0



Every reconstructed feature matches the saved Notebook 3 historical evidence.

Feature reconstruction passed.
All 129 features reproduce the historical Notebook 3 evidence, and 344 provisional 2023 rows exist in memory.
Nothing has been saved and no 2024 risk score has been generated.


In [6]:
# ============================================================
# CORRECTION: RESTORE YEAR AS AN AUDITABLE FEATURE COLUMN
# ============================================================

historical_reference["Year"] = (
    historical_reference.index
    .get_level_values("Year")
    .astype("int16")
)

assert "Year" in historical_reference.columns

assert historical_reference["Year"].min() == 2010
assert historical_reference["Year"].max() == 2022

print(
    "Year restored as a historical feature column. "
    "Step 4 can now be rerun."
)

Year restored as a historical feature column. Step 4 can now be rerun.


In [7]:
# ============================================================
# NOTEBOOK 5 — STEP 4
# DEFINE 2023 SCORING ELIGIBILITY AND GENERATE 2024 RISKS
#
# These are model-generated early-warning estimates, not
# observed 2024 shortage events.
# ============================================================

from pathlib import Path
import hashlib
import json
import cloudpickle
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Retrieve all 2023 reconstructed features
# ------------------------------------------------------------

features_2023_all = (
    reconstructed_features.loc[
        reconstructed_features["Year"].eq(2023)
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(features_2023_all) == 344
assert features_2023_all["Area"].nunique() == 43
assert features_2023_all["Item Code"].nunique() == 8
assert features_2023_all.columns.tolist() == (
    locked_feature_names
)


# ------------------------------------------------------------
# 2. Attach descriptive and quality metadata
# ------------------------------------------------------------

panel_metadata = (
    panel_for_features
    .reset_index()
)

metadata_candidates = [
    "Area",
    "Item Code",
    "Year",
    "Area Code",
    "Area Code (M49)",
    "M49 Code",
    "Item Code (FBS)",
    "Item",
    "food_supply_kcal_cap_day",
    "food_supply_kcal_cap_day_source_present",
    "food_supply_kcal_cap_day_missing",
    "recorded_element_count",
    "source_absent_element_count",
    "source_present_missing_element_count",
    "recorded_zero_element_count",
    "recorded_negative_element_count",
    "supply_outcome_recorded_count",
    "supply_outcomes_complete",
    "production_status",
    "negative_domestic_supply_flag",
]

available_metadata_columns = [
    column
    for column in metadata_candidates
    if column in panel_metadata.columns
]

metadata_2023 = (
    panel_metadata.loc[
        panel_metadata["Year"].eq(2023),
        available_metadata_columns,
    ]
    .copy()
)

scoring_register_2024 = (
    features_2023_all
    .merge(
        metadata_2023,
        on=["Area", "Item Code", "Year"],
        how="left",
        validate="one_to_one",
        suffixes=("", "_metadata"),
    )
)

assert len(scoring_register_2024) == 344


# ------------------------------------------------------------
# 3. Define prospective scoring eligibility
# ------------------------------------------------------------
# The historical outcome definition required:
# - current calorie availability of at least 5 kcal/person/day;
# - a sufficiently large decline in the following year.
#
# For 2023, the following-year value is unknown by design.
# Eligibility can therefore use only information available
# in 2023: the current calorie value must be recorded and must
# meet the locked minimum-current-value rule.

minimum_current_kcal = float(
    feature_specification[
        "target_definition"
    ][
        "minimum_current_kcal_capita_day"
    ]
)

current_kcal = (
    scoring_register_2024[
        "food_supply_kcal_cap_day"
    ]
)

scoring_register_2024[
    "eligible_for_2024_scoring"
] = (
    current_kcal.notna()
    & current_kcal.ge(minimum_current_kcal)
)

scoring_register_2024[
    "scoring_exclusion_reason"
] = np.select(
    [
        current_kcal.isna(),
        current_kcal.lt(
            minimum_current_kcal
        ),
    ],
    [
        (
            "Current 2023 calorie availability "
            "is missing"
        ),
        (
            "Current 2023 calorie availability "
            "is below 5 kcal/person/day"
        ),
    ],
    default="Eligible for prospective scoring",
)


# ------------------------------------------------------------
# 4. Validate the prospective eligibility rule historically
# ------------------------------------------------------------
# Every labelled historical observation must satisfy the
# current-year part of the target definition. Equality is not
# expected because some otherwise eligible current rows lacked
# a usable following-year outcome.

historical_panel_aligned = (
    panel_for_features.reindex(
        historical_reference.index
    )
)

historical_current_rule = (
    historical_panel_aligned[
        "food_supply_kcal_cap_day"
    ]
    .notna()
    & historical_panel_aligned[
        "food_supply_kcal_cap_day"
    ]
    .ge(minimum_current_kcal)
)

assert historical_current_rule.all(), (
    "The proposed prospective eligibility rule is "
    "inconsistent with one or more historical labelled rows."
)


# ------------------------------------------------------------
# 5. Audit the 2023 feature evidence
# ------------------------------------------------------------

numeric_feature_names = [
    feature
    for feature in locked_feature_names
    if feature not in categorical_features
]

scoring_register_2024[
    "missing_locked_feature_count"
] = (
    scoring_register_2024[
        numeric_feature_names
    ]
    .isna()
    .sum(axis=1)
    .astype("int16")
)

training_numeric_minimum = (
    historical_reference[
        numeric_feature_names
    ]
    .min(axis=0, skipna=True)
)

training_numeric_maximum = (
    historical_reference[
        numeric_feature_names
    ]
    .max(axis=0, skipna=True)
)

below_training_range = (
    scoring_register_2024[
        numeric_feature_names
    ]
    .lt(
        training_numeric_minimum,
        axis="columns",
    )
)

above_training_range = (
    scoring_register_2024[
        numeric_feature_names
    ]
    .gt(
        training_numeric_maximum,
        axis="columns",
    )
)

scoring_register_2024[
    "numeric_features_outside_training_range_count"
] = (
    below_training_range
    | above_training_range
).sum(axis=1).astype("int16")


# ------------------------------------------------------------
# 6. Resolve the domestic-supply denominator ambiguity
# ------------------------------------------------------------
# The historical comparison could not distinguish signed from
# absolute domestic-supply denominators. This only matters if
# an eligible 2023 observation has negative domestic supply.

eligible_mask = (
    scoring_register_2024[
        "eligible_for_2024_scoring"
    ]
)

eligible_negative_domestic_supply = (
    eligible_mask
    & scoring_register_2024[
        "domestic_supply_1000t"
    ].lt(0)
)

denominator_ambiguity_summary = pd.DataFrame(
    {
        "Check": [
            "Eligible 2023 observations",
            "Eligible rows with negative domestic supply",
            "Eligible rows with zero domestic supply",
            "Eligible rows with missing domestic supply",
        ],
        "Rows": [
            int(eligible_mask.sum()),
            int(
                eligible_negative_domestic_supply.sum()
            ),
            int(
                (
                    eligible_mask
                    & scoring_register_2024[
                        "domestic_supply_1000t"
                    ].eq(0)
                ).sum()
            ),
            int(
                (
                    eligible_mask
                    & scoring_register_2024[
                        "domestic_supply_1000t"
                    ].isna()
                ).sum()
            ),
        ],
    }
)

print("Domestic-supply denominator audit:")
display(denominator_ambiguity_summary)

if eligible_negative_domestic_supply.any():

    display(
        scoring_register_2024.loc[
            eligible_negative_domestic_supply,
            [
                "Area",
                "Item Code",
                "Item",
                "Year",
                "domestic_supply_1000t",
                "food_supply_kcal_cap_day",
            ],
        ]
    )

assert not eligible_negative_domestic_supply.any(), (
    "At least one eligible 2023 row has negative "
    "domestic supply. Stop before scoring because the "
    "saved evidence does not uniquely identify the "
    "intended denominator convention."
)


# ------------------------------------------------------------
# 7. Confirm all categories were represented in training
# ------------------------------------------------------------

unknown_area = ~scoring_register_2024[
    "Area"
].isin(
    historical_reference.index
    .get_level_values("Area")
    .unique()
)

unknown_item = ~scoring_register_2024[
    "Item Code"
].isin(
    historical_reference.index
    .get_level_values("Item Code")
    .unique()
)

scoring_register_2024[
    "unknown_model_category_count"
] = (
    unknown_area.astype(int)
    + unknown_item.astype(int)
).astype("int8")

assert not unknown_area.any()
assert not unknown_item.any()


# ------------------------------------------------------------
# 8. Load and fingerprint-check the operational model
# ------------------------------------------------------------

operational_model_path = (
    model_output_directory
    / "africa_shortage_operational_model_2010_2022.pkl"
)

manifest_path = (
    model_output_directory
    / "shortage_operational_model_manifest.json"
)

with open(
    manifest_path,
    "r",
    encoding="utf-8",
) as manifest_file:
    operational_manifest = json.load(
        manifest_file
    )


def calculate_sha256(file_path):

    sha256_hash = hashlib.sha256()

    with open(file_path, "rb") as input_file:

        for block in iter(
            lambda: input_file.read(
                1024 * 1024
            ),
            b"",
        ):
            sha256_hash.update(block)

    return sha256_hash.hexdigest()


current_model_signature = calculate_sha256(
    operational_model_path
)

assert current_model_signature == (
    operational_manifest[
        "model_file_signature_sha256"
    ]
), (
    "The operational model file fingerprint does not "
    "match its saved manifest."
)

with open(
    operational_model_path,
    "rb",
) as model_file:
    operational_model_bundle = (
        cloudpickle.load(model_file)
    )

operational_pipeline = (
    operational_model_bundle[
        "fitted_pipeline"
    ]
)

bundle_feature_names = (
    operational_model_bundle[
        "feature_names"
    ]
)

probability_threshold = float(
    operational_model_bundle[
        "probability_threshold"
    ]
)

assert bundle_feature_names == locked_feature_names
assert probability_threshold == 0.13
assert operational_manifest[
    "probability_threshold"
] == 0.13


# ------------------------------------------------------------
# 9. Generate probabilities for eligible rows only
# ------------------------------------------------------------

scoring_register_2024[
    "predicted_shortage_probability"
] = np.nan

eligible_feature_table = (
    scoring_register_2024.loc[
        eligible_mask,
        locked_feature_names,
    ]
    .copy()
)

eligible_probabilities = (
    operational_pipeline.predict_proba(
        eligible_feature_table
    )[:, 1]
)

assert len(eligible_probabilities) == int(
    eligible_mask.sum()
)

assert np.isfinite(
    eligible_probabilities
).all()

assert (
    (eligible_probabilities >= 0)
    & (eligible_probabilities <= 1)
).all()

scoring_register_2024.loc[
    eligible_mask,
    "predicted_shortage_probability",
] = eligible_probabilities


# ------------------------------------------------------------
# 10. Assign operational risk bands
# ------------------------------------------------------------

risk_band_labels = [
    "Very low: below 5%",
    "Watch: 5% to below 13%",
    "Warning: 13% to below 30%",
    "High warning: 30% or more",
]

scoring_register_2024[
    "risk_band"
] = pd.cut(
    scoring_register_2024[
        "predicted_shortage_probability"
    ],
    bins=[
        -np.inf,
        0.05,
        probability_threshold,
        0.30,
        np.inf,
    ],
    labels=risk_band_labels,
    right=False,
    ordered=True,
)

scoring_register_2024[
    "predicted_shortage_warning"
] = pd.Series(
    pd.NA,
    index=scoring_register_2024.index,
    dtype="Int8",
)

scoring_register_2024.loc[
    eligible_mask,
    "predicted_shortage_warning",
] = (
    scoring_register_2024.loc[
        eligible_mask,
        "predicted_shortage_probability",
    ]
    .ge(probability_threshold)
    .astype("int8")
)

scoring_register_2024[
    "target_year"
] = pd.Series(
    2024,
    index=scoring_register_2024.index,
    dtype="int16",
)

scoring_register_2024[
    "result_type"
] = np.where(
    eligible_mask,
    (
        "Model-generated 2024 early-warning "
        "estimate"
    ),
    "Not scored",
)


# ------------------------------------------------------------
# 11. Validate warning logic
# ------------------------------------------------------------

scored_mask = (
    scoring_register_2024[
        "predicted_shortage_probability"
    ].notna()
)

assert scored_mask.equals(eligible_mask)

assert scoring_register_2024.loc[
    scored_mask,
    "risk_band",
].notna().all()

assert scoring_register_2024.loc[
    ~scored_mask,
    "risk_band",
].isna().all()

assert (
    scoring_register_2024.loc[
        scored_mask,
        "predicted_shortage_warning",
    ].astype(int)
    == scoring_register_2024.loc[
        scored_mask,
        "predicted_shortage_probability",
    ].ge(probability_threshold).astype(int)
).all()


# ------------------------------------------------------------
# 12. Produce audit summaries
# ------------------------------------------------------------

eligibility_summary = (
    scoring_register_2024
    .groupby(
        "scoring_exclusion_reason",
        dropna=False,
        observed=True,
    )
    .agg(
        Rows=("Area", "size"),
        Countries=("Area", "nunique"),
        Commodities=("Item Code", "nunique"),
    )
    .reset_index()
    .sort_values(
        "Rows",
        ascending=False,
    )
)

risk_band_summary_2024 = (
    scoring_register_2024.loc[
        scored_mask
    ]
    .groupby(
        "risk_band",
        observed=False,
    )
    .agg(
        Observations=(
            "predicted_shortage_probability",
            "size",
        ),
        Countries=("Area", "nunique"),
        Commodities=("Item Code", "nunique"),
        Average_probability=(
            "predicted_shortage_probability",
            "mean",
        ),
        Minimum_probability=(
            "predicted_shortage_probability",
            "min",
        ),
        Maximum_probability=(
            "predicted_shortage_probability",
            "max",
        ),
    )
    .reset_index()
)

preliminary_country_warning_summary = (
    scoring_register_2024.loc[
        scored_mask
    ]
    .groupby(
        "Area",
        observed=True,
    )
    .agg(
        Eligible_commodities=(
            "Item Code",
            "size",
        ),
        Watch_or_higher=(
            "predicted_shortage_probability",
            lambda values: int(
                values.ge(0.05).sum()
            ),
        ),
        Formal_warnings=(
            "predicted_shortage_warning",
            "sum",
        ),
        High_warnings=(
            "predicted_shortage_probability",
            lambda values: int(
                values.ge(0.30).sum()
            ),
        ),
        Average_probability=(
            "predicted_shortage_probability",
            "mean",
        ),
        Maximum_probability=(
            "predicted_shortage_probability",
            "max",
        ),
        Missing_feature_average=(
            "missing_locked_feature_count",
            "mean",
        ),
        Outside_training_range_average=(
            "numeric_features_outside_training_range_count",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "Formal_warnings",
            "High_warnings",
            "Maximum_probability",
        ],
        ascending=[False, False, False],
    )
)

preliminary_commodity_warning_summary = (
    scoring_register_2024.loc[
        scored_mask
    ]
    .groupby(
        ["Item Code", "Item"],
        observed=True,
    )
    .agg(
        Eligible_countries=("Area", "size"),
        Watch_or_higher=(
            "predicted_shortage_probability",
            lambda values: int(
                values.ge(0.05).sum()
            ),
        ),
        Formal_warnings=(
            "predicted_shortage_warning",
            "sum",
        ),
        High_warnings=(
            "predicted_shortage_probability",
            lambda values: int(
                values.ge(0.30).sum()
            ),
        ),
        Average_probability=(
            "predicted_shortage_probability",
            "mean",
        ),
        Maximum_probability=(
            "predicted_shortage_probability",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "Formal_warnings",
            "High_warnings",
            "Maximum_probability",
        ],
        ascending=[False, False, False],
    )
)

highest_2024_risks = (
    scoring_register_2024.loc[
        scored_mask,
        [
            "Area",
            "Item Code",
            "Item",
            "Year",
            "target_year",
            "food_supply_kcal_cap_day",
            "predicted_shortage_probability",
            "risk_band",
            "predicted_shortage_warning",
            "missing_locked_feature_count",
            "numeric_features_outside_training_range_count",
        ],
    ]
    .sort_values(
        "predicted_shortage_probability",
        ascending=False,
    )
    .head(30)
)


# ------------------------------------------------------------
# 13. Display results
# ------------------------------------------------------------

print("Prospective 2023 scoring eligibility:")
display(eligibility_summary)

print("\nProvisional 2024 risk-band summary:")
display(
    risk_band_summary_2024.style.format(
        {
            "Average_probability": "{:.4f}",
            "Minimum_probability": "{:.4f}",
            "Maximum_probability": "{:.4f}",
        }
    )
)

print(
    "\nHighest model-generated 2024 risks "
    "(not observed events):"
)
display(
    highest_2024_risks.style.format(
        {
            "food_supply_kcal_cap_day":
                "{:.2f}",
            "predicted_shortage_probability":
                "{:.4f}",
        }
    )
)

print(
    "\nPreliminary country warning summary "
    "(not yet a country ranking):"
)
display(
    preliminary_country_warning_summary.style.format(
        {
            "Average_probability": "{:.4f}",
            "Maximum_probability": "{:.4f}",
            "Missing_feature_average": "{:.2f}",
            "Outside_training_range_average":
                "{:.2f}",
        }
    )
)

print("\nPreliminary commodity warning summary:")
display(
    preliminary_commodity_warning_summary.style.format(
        {
            "Average_probability": "{:.4f}",
            "Maximum_probability": "{:.4f}",
        }
    )
)

print(
    "\nThe 2024 risk estimates were generated in memory.\n"
    "They have not yet been saved or treated as "
    "observed shortage events."
)

Domestic-supply denominator audit:


,Check,Rows
0,Eligible 2023 observations,249
1,Eligible rows with negative domestic supply,0
2,Eligible rows with zero domestic supply,0
3,Eligible rows with missing domestic supply,0


Prospective 2023 scoring eligibility:


,scoring_exclusion_reason,Rows,Countries,Commodities
2,Eligible for prospective scoring,249,43,8
0,Current 2023 calorie availability is below 5 k...,66,32,6
1,Current 2023 calorie availability is missing,29,21,4



Provisional 2024 risk-band summary:


...


Preliminary country warning summary (not yet a country ranking):


,Area,Eligible_commodities,Watch_or_higher,Formal_warnings,High_warnings,Average_probability,Maximum_probability,Missing_feature_average,Outside_training_range_average
39,Uganda,7,7,5,1,0.1962,0.3182,5.29,1.00
24,Malawi,7,6,4,2,0.2449,0.6106,6.00,1.14
36,Sierra Leone,7,4,4,0,0.1346,0.2914,11.00,1.00
42,Zimbabwe,7,5,4,0,0.1220,0.2001,7.29,1.00
15,Gambia,7,4,3,2,0.1484,0.3597,13.86,1.00
26,Mauritius,5,4,3,1,0.1777,0.4445,15.00,1.60
35,Seychelles,4,4,3,1,0.2338,0.3979,27.75,1.00
20,Lesotho,4,3,3,1,0.1834,0.3235,11.25,1.00
40,United Republic of Tanzania,7,6,3,0,0.1074,0.2236,3.71,1.14
22,Libya,4,4,3,0,0.1685,0.2193,13.00,1.00



Preliminary commodity warning summary:


,Item Code,Item,Eligible_countries,Watch_or_higher,Formal_warnings,High_warnings,Average_probability,Maximum_probability
6,2552,Groundnuts,35,28,20,1,0.1584,0.5291
0,2511,Wheat and products,43,29,16,5,0.1313,0.6106
7,2807,Rice and products,43,29,10,0,0.0916,0.2514
3,2518,Sorghum and products,26,21,9,1,0.1153,0.3235
1,2514,Maize and products,41,20,8,4,0.0989,0.6195
2,2517,Millet and products,21,14,7,1,0.1110,0.3234
4,2532,Cassava and products,28,10,6,0,0.0664,0.2111
5,2535,Yams,12,5,1,0,0.0588,0.2405



The 2024 risk estimates were generated in memory.
They have not yet been saved or treated as observed shortage events.


In [8]:
# ============================================================
# NOTEBOOK 5 — STEP 5
# AUDIT DATA SUPPORT AND FORWARD-SCORE STABILITY
#
# This distinguishes model risk from the strength of evidence
# supporting each individual risk estimate.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Calculate historical missingness reference boundaries
# ------------------------------------------------------------

historical_missing_feature_count = (
    historical_reference[
        numeric_feature_names
    ]
    .isna()
    .sum(axis=1)
)

historical_missing_q75 = float(
    historical_missing_feature_count.quantile(
        0.75
    )
)

historical_missing_q95 = float(
    historical_missing_feature_count.quantile(
        0.95
    )
)

historical_missing_reference = pd.DataFrame(
    {
        "Measure": [
            "Historical observations",
            "Minimum missing numeric features",
            "Median missing numeric features",
            "75th percentile",
            "95th percentile",
            "Maximum missing numeric features",
        ],
        "Value": [
            len(historical_missing_feature_count),
            historical_missing_feature_count.min(),
            historical_missing_feature_count.median(),
            historical_missing_q75,
            historical_missing_q95,
            historical_missing_feature_count.max(),
        ],
    }
)


# ------------------------------------------------------------
# 2. Remove the expected Year=2023 range departure
# ------------------------------------------------------------

non_year_numeric_features = [
    feature
    for feature in numeric_feature_names
    if feature != "Year"
]

below_training_range_non_year = (
    scoring_register_2024[
        non_year_numeric_features
    ]
    .lt(
        training_numeric_minimum[
            non_year_numeric_features
        ],
        axis="columns",
    )
)

above_training_range_non_year = (
    scoring_register_2024[
        non_year_numeric_features
    ]
    .gt(
        training_numeric_maximum[
            non_year_numeric_features
        ],
        axis="columns",
    )
)

scoring_register_2024[
    "non_year_features_outside_training_range_count"
] = (
    below_training_range_non_year
    | above_training_range_non_year
).sum(axis=1).astype("int16")

# Confirm that the earlier count included Year for every
# eligible row.
year_outside_training_range = (
    scoring_register_2024["Year"]
    > training_numeric_maximum["Year"]
)

assert year_outside_training_range.loc[
    eligible_mask
].all()

assert (
    scoring_register_2024.loc[
        eligible_mask,
        "numeric_features_outside_training_range_count",
    ]
    ==
    (
        scoring_register_2024.loc[
            eligible_mask,
            "non_year_features_outside_training_range_count",
        ]
        + 1
    )
).all()


# ------------------------------------------------------------
# 3. Attach country-commodity historical support
# ------------------------------------------------------------

historical_support_source = (
    model_ready_history.copy()
)

historical_support_source["Area"] = (
    historical_support_source[
        "Area"
    ].astype(str)
)

historical_support_source["Item Code"] = (
    pd.to_numeric(
        historical_support_source["Item Code"],
        errors="raise",
    )
    .astype("int64")
    .astype(str)
)

pair_historical_support = (
    historical_support_source
    .groupby(
        ["Area", "Item Code"],
        observed=True,
    )
    .agg(
        pair_historical_rows=(
            "shortage_next_year",
            "size",
        ),
        pair_historical_shortages=(
            "shortage_next_year",
            "sum",
        ),
        pair_predictor_year_start=(
            "Year",
            "min",
        ),
        pair_predictor_year_end=(
            "Year",
            "max",
        ),
    )
    .reset_index()
)

pair_historical_support[
    "pair_historical_shortage_rate"
] = (
    pair_historical_support[
        "pair_historical_shortages"
    ]
    / pair_historical_support[
        "pair_historical_rows"
    ]
)

scoring_register_2024 = (
    scoring_register_2024
    .merge(
        pair_historical_support,
        on=["Area", "Item Code"],
        how="left",
        validate="many_to_one",
    )
)

pair_support_columns = [
    "pair_historical_rows",
    "pair_historical_shortages",
    "pair_predictor_year_start",
    "pair_predictor_year_end",
]

scoring_register_2024[
    pair_support_columns
] = (
    scoring_register_2024[
        pair_support_columns
    ]
    .fillna(0)
)

scoring_register_2024[
    "pair_has_historical_training_support"
] = (
    scoring_register_2024[
        "pair_historical_rows"
    ].gt(0)
)


# ------------------------------------------------------------
# 4. Assign a transparent evidence-review level
# ------------------------------------------------------------
# These labels do not change probabilities or warning status.
#
# Limited evidence:
# - no eligible historical rows for the same pair; or
# - missingness exceeds 95% of historical training rows.
#
# Heightened review:
# - missingness exceeds the historical 75th percentile; or
# - any non-calendar feature lies outside its training range; or
# - fewer than five historical rows exist for the same pair.
#
# Otherwise, the row is within usual historical support.

limited_evidence = (
    ~scoring_register_2024[
        "pair_has_historical_training_support"
    ]
    |
    scoring_register_2024[
        "missing_locked_feature_count"
    ].gt(historical_missing_q95)
)

heightened_review = (
    scoring_register_2024[
        "missing_locked_feature_count"
    ].gt(historical_missing_q75)
    |
    scoring_register_2024[
        "non_year_features_outside_training_range_count"
    ].gt(0)
    |
    scoring_register_2024[
        "pair_historical_rows"
    ].lt(5)
)

scoring_register_2024[
    "evidence_review_level"
] = np.select(
    [
        limited_evidence,
        heightened_review,
    ],
    [
        "Limited evidence: priority manual review",
        "Heightened evidence review",
    ],
    default="Within usual historical support",
)

scoring_register_2024.loc[
    ~eligible_mask,
    "evidence_review_level",
] = "Not scored"


# ------------------------------------------------------------
# 5. Compare 2024 score distribution with the locked test
# ------------------------------------------------------------

final_test_predictions_path = (
    model_output_directory
    / "shortage_final_test_predictions.parquet"
)

locked_test_predictions = pd.read_parquet(
    final_test_predictions_path
)

required_test_columns = {
    "predicted_shortage_probability",
}

assert required_test_columns.issubset(
    locked_test_predictions.columns
)

locked_test_probabilities = (
    locked_test_predictions[
        "predicted_shortage_probability"
    ].astype(float)
)

forward_probabilities = (
    scoring_register_2024.loc[
        eligible_mask,
        "predicted_shortage_probability",
    ].astype(float)
)

locked_test_risk_band = pd.cut(
    locked_test_probabilities,
    bins=[
        -np.inf,
        0.05,
        probability_threshold,
        0.30,
        np.inf,
    ],
    labels=risk_band_labels,
    right=False,
    ordered=True,
)

forward_risk_band = (
    scoring_register_2024.loc[
        eligible_mask,
        "risk_band",
    ]
)

test_band_counts = (
    locked_test_risk_band
    .value_counts(sort=False)
    .reindex(risk_band_labels, fill_value=0)
)

forward_band_counts = (
    forward_risk_band
    .value_counts(sort=False)
    .reindex(risk_band_labels, fill_value=0)
)

risk_distribution_comparison = pd.DataFrame(
    {
        "Risk band": risk_band_labels,
        "Locked test observations":
            test_band_counts.to_numpy(),
        "Locked test share":
            (
                test_band_counts
                / test_band_counts.sum()
            ).to_numpy(),
        "2024 scored observations":
            forward_band_counts.to_numpy(),
        "2024 scored share":
            (
                forward_band_counts
                / forward_band_counts.sum()
            ).to_numpy(),
    }
)

risk_distribution_comparison[
    "Share difference"
] = (
    risk_distribution_comparison[
        "2024 scored share"
    ]
    - risk_distribution_comparison[
        "Locked test share"
    ]
)


# Population Stability Index is used here only as a descriptive
# distribution-shift diagnostic, not as a performance measure.

small_value = 1e-6

test_shares_for_psi = (
    risk_distribution_comparison[
        "Locked test share"
    ].clip(lower=small_value)
)

forward_shares_for_psi = (
    risk_distribution_comparison[
        "2024 scored share"
    ].clip(lower=small_value)
)

risk_band_psi = float(
    (
        (
            forward_shares_for_psi
            - test_shares_for_psi
        )
        * np.log(
            forward_shares_for_psi
            / test_shares_for_psi
        )
    ).sum()
)


# ------------------------------------------------------------
# 6. Create overall score-distribution summary
# ------------------------------------------------------------

score_distribution_summary = pd.DataFrame(
    {
        "Measure": [
            "Observations",
            "Mean probability",
            "Median probability",
            "90th percentile probability",
            "Maximum probability",
            "Watch-or-higher share",
            "Formal-warning share",
            "High-warning share",
        ],
        "Locked test": [
            len(locked_test_probabilities),
            locked_test_probabilities.mean(),
            locked_test_probabilities.median(),
            locked_test_probabilities.quantile(0.90),
            locked_test_probabilities.max(),
            locked_test_probabilities.ge(0.05).mean(),
            locked_test_probabilities.ge(
                probability_threshold
            ).mean(),
            locked_test_probabilities.ge(0.30).mean(),
        ],
        "2024 scoring register": [
            len(forward_probabilities),
            forward_probabilities.mean(),
            forward_probabilities.median(),
            forward_probabilities.quantile(0.90),
            forward_probabilities.max(),
            forward_probabilities.ge(0.05).mean(),
            forward_probabilities.ge(
                probability_threshold
            ).mean(),
            forward_probabilities.ge(0.30).mean(),
        ],
    }
)

score_distribution_summary[
    "Difference"
] = (
    score_distribution_summary[
        "2024 scoring register"
    ]
    - score_distribution_summary[
        "Locked test"
    ]
)


# ------------------------------------------------------------
# 7. Summarise evidence support
# ------------------------------------------------------------

evidence_support_summary = (
    scoring_register_2024.loc[
        eligible_mask
    ]
    .groupby(
        "evidence_review_level",
        observed=True,
    )
    .agg(
        Scored_observations=("Area", "size"),
        Countries=("Area", "nunique"),
        Formal_warnings=(
            "predicted_shortage_warning",
            "sum",
        ),
        High_warnings=(
            "predicted_shortage_probability",
            lambda values: int(
                values.ge(0.30).sum()
            ),
        ),
        Average_probability=(
            "predicted_shortage_probability",
            "mean",
        ),
        Average_missing_features=(
            "missing_locked_feature_count",
            "mean",
        ),
        Average_non_year_outside_range=(
            "non_year_features_outside_training_range_count",
            "mean",
        ),
    )
    .reset_index()
)

evidence_support_summary[
    "Share_of_scored_observations"
] = (
    evidence_support_summary[
        "Scored_observations"
    ]
    / int(eligible_mask.sum())
)


# ------------------------------------------------------------
# 8. Audit the 12 high-warning observations
# ------------------------------------------------------------

high_warning_evidence_audit = (
    scoring_register_2024.loc[
        eligible_mask
        & scoring_register_2024[
            "predicted_shortage_probability"
        ].ge(0.30),
        [
            "Area",
            "Item Code",
            "Item",
            "target_year",
            "food_supply_kcal_cap_day",
            "predicted_shortage_probability",
            "risk_band",
            "evidence_review_level",
            "missing_locked_feature_count",
            "non_year_features_outside_training_range_count",
            "pair_historical_rows",
            "pair_historical_shortages",
            "pair_historical_shortage_rate",
        ],
    ]
    .sort_values(
        "predicted_shortage_probability",
        ascending=False,
    )
)

limited_support_formal_warnings = (
    scoring_register_2024.loc[
        eligible_mask
        & scoring_register_2024[
            "predicted_shortage_warning"
        ].eq(1)
        & scoring_register_2024[
            "evidence_review_level"
        ].eq(
            "Limited evidence: priority manual review"
        ),
        [
            "Area",
            "Item Code",
            "Item",
            "predicted_shortage_probability",
            "risk_band",
            "missing_locked_feature_count",
            "non_year_features_outside_training_range_count",
            "pair_historical_rows",
            "pair_historical_shortages",
        ],
    ]
    .sort_values(
        "predicted_shortage_probability",
        ascending=False,
    )
)


# ------------------------------------------------------------
# 9. Display the audit
# ------------------------------------------------------------

print("Historical missingness reference:")
display(historical_missing_reference)

print(
    "\nLocked-test versus forward-score distribution:"
)
display(
    score_distribution_summary.style.format(
        {
            "Locked test": "{:.4f}",
            "2024 scoring register": "{:.4f}",
            "Difference": "{:+.4f}",
        },
        subset=pd.IndexSlice[
            score_distribution_summary[
                "Measure"
            ].ne("Observations"),
            [
                "Locked test",
                "2024 scoring register",
                "Difference",
            ],
        ],
    )
)

print("\nRisk-band distribution comparison:")
display(
    risk_distribution_comparison.style.format(
        {
            "Locked test share": "{:.2%}",
            "2024 scored share": "{:.2%}",
            "Share difference": "{:+.2%}",
        }
    )
)

print(
    "\nRisk-band Population Stability Index:",
    f"{risk_band_psi:.4f}",
)
print(
    "This is a descriptive shift diagnostic, "
    "not a measure of predictive accuracy."
)

print("\nEvidence-support summary:")
display(
    evidence_support_summary.style.format(
        {
            "Average_probability": "{:.4f}",
            "Average_missing_features": "{:.2f}",
            "Average_non_year_outside_range":
                "{:.2f}",
            "Share_of_scored_observations":
                "{:.2%}",
        }
    )
)

print("\nEvidence audit of all high warnings:")
display(
    high_warning_evidence_audit.style.format(
        {
            "food_supply_kcal_cap_day": "{:.2f}",
            "predicted_shortage_probability":
                "{:.4f}",
            "pair_historical_shortage_rate":
                "{:.2%}",
        }
    )
)

print(
    "\nFormal warnings with limited evidence support:"
)

if limited_support_formal_warnings.empty:
    print("None.")
else:
    display(
        limited_support_formal_warnings.style.format(
            {
                "predicted_shortage_probability":
                    "{:.4f}",
            }
        )
    )

print(
    "\nForward-score support audit completed.\n"
    "No probability or warning threshold was changed, "
    "and no output was saved."
)

Historical missingness reference:


,Measure,Value
0,Historical observations,3248.0
1,Minimum missing numeric features,0.0
2,Median missing numeric features,9.0
3,75th percentile,19.0
4,95th percentile,73.0
5,Maximum missing numeric features,86.0



Locked-test versus forward-score distribution:


,Measure,Locked test,2024 scoring register,Difference
0,Observations,498.000000,249.000000,-249.000000
1,Mean probability,0.1206,0.1088,-0.0118
2,Median probability,0.0794,0.0800,+0.0006
3,90th percentile probability,0.2574,0.2285,-0.0289
4,Maximum probability,0.7312,0.6195,-0.1117
5,Watch-or-higher share,0.6747,0.6265,-0.0482
6,Formal-warning share,0.3193,0.3092,-0.0100
7,High-warning share,0.0823,0.0482,-0.0341



Risk-band distribution comparison:


...


Evidence audit of all high warnings:


,Area,Item Code,Item,target_year,food_supply_kcal_cap_day,predicted_shortage_probability,risk_band,evidence_review_level,missing_locked_feature_count,non_year_features_outside_training_range_count,pair_historical_rows,pair_historical_shortages,pair_historical_shortage_rate
57,Congo,2514,Maize and products,2024,65.32,0.6195,High warning: 30% or more,Within usual historical support,5,0,13.000000,3.000000,23.08%
192,Malawi,2511,Wheat and products,2024,42.07,0.6106,High warning: 30% or more,Within usual historical support,4,0,13.000000,7.000000,53.85%
198,Malawi,2552,Groundnuts,2024,27.64,0.5291,High warning: 30% or more,Within usual historical support,9,0,5.000000,4.000000,80.00%
256,Rwanda,2511,Wheat and products,2024,121.35,0.5035,High warning: 30% or more,Within usual historical support,5,0,13.000000,4.000000,30.77%
209,Mauritius,2514,Maize and products,2024,38.65,0.4445,High warning: 30% or more,Within usual historical support,3,0,13.000000,3.000000,23.08%
144,Guinea-Bissau,2511,Wheat and products,2024,79.42,0.4132,High warning: 30% or more,Heightened evidence review,31,0,13.000000,1.000000,7.69%
281,Seychelles,2514,Maize and products,2024,191.61,0.3979,High warning: 30% or more,Heightened evidence review,25,0,13.000000,4.000000,30.77%
168,Liberia,2511,Wheat and products,2024,25.23,0.3637,High warning: 30% or more,Heightened evidence review,27,0,13.000000,6.000000,46.15%
121,Gambia,2514,Maize and products,2024,80.18,0.3597,High warning: 30% or more,Within usual historical support,7,0,13.000000,4.000000,30.77%
163,Lesotho,2518,Sorghum and products,2024,11.42,0.3235,High warning: 30% or more,Within usual historical support,9,0,13.000000,6.000000,46.15%



Formal warnings with limited evidence support:


,Area,Item Code,Item,predicted_shortage_probability,risk_band,missing_locked_feature_count,non_year_features_outside_training_range_count,pair_historical_rows,pair_historical_shortages
212,Mauritius,2532,Cassava and products,0.1862,Warning: 13% to below 30%,26,3,0.000000,0.000000



Forward-score support audit completed.
No probability or warning threshold was changed, and no output was saved.


In [9]:
# ============================================================
# NOTEBOOK 5 — STEP 6
# SAVE THE VERIFIED 2023 FEATURES AND 2024 RISK REGISTER
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Define ranking eligibility without changing warnings
# ------------------------------------------------------------

formal_warning_mask = (
    eligible_mask
    & scoring_register_2024[
        "predicted_shortage_warning"
    ]
    .fillna(0)
    .eq(1)
)

high_warning_mask = (
    eligible_mask
    & scoring_register_2024[
        "predicted_shortage_probability"
    ].ge(0.30)
)

watch_or_higher_mask = (
    eligible_mask
    & scoring_register_2024[
        "predicted_shortage_probability"
    ].ge(0.05)
)

limited_evidence_mask = (
    scoring_register_2024[
        "evidence_review_level"
    ].eq(
        "Limited evidence: priority manual review"
    )
)

scoring_register_2024[
    "eligible_for_headline_ranking"
] = (
    eligible_mask
    & ~limited_evidence_mask
)

scoring_register_2024[
    "headline_ranking_exclusion_reason"
] = np.select(
    [
        ~eligible_mask,
        limited_evidence_mask,
    ],
    [
        "Observation was not eligible for scoring",
        (
            "Limited evidence support; retained in the "
            "warning register but excluded from headline "
            "ranking calculations"
        ),
    ],
    default="Eligible for headline ranking inputs",
)

assert scoring_register_2024[
    "eligible_for_headline_ranking"
].sum() == 248

assert formal_warning_mask.sum() == 77
assert high_warning_mask.sum() == 12

assert (
    formal_warning_mask
    & limited_evidence_mask
).sum() == 1


# ------------------------------------------------------------
# 2. Create the canonical row-level register
# ------------------------------------------------------------

preferred_register_columns = [
    "Area Code",
    "Area Code (M49)",
    "M49 Code",
    "Area",
    "Item Code (FBS)",
    "Item Code",
    "Item",
    "Year",
    "target_year",
    "result_type",
    "food_supply_kcal_cap_day",
    "eligible_for_2024_scoring",
    "scoring_exclusion_reason",
    "predicted_shortage_probability",
    "risk_band",
    "predicted_shortage_warning",
    "evidence_review_level",
    "eligible_for_headline_ranking",
    "headline_ranking_exclusion_reason",
    "missing_locked_feature_count",
    "numeric_features_outside_training_range_count",
    "non_year_features_outside_training_range_count",
    "unknown_model_category_count",
    "pair_has_historical_training_support",
    "pair_historical_rows",
    "pair_historical_shortages",
    "pair_historical_shortage_rate",
    "pair_predictor_year_start",
    "pair_predictor_year_end",
    "production_status",
    "recorded_element_count",
    "source_absent_element_count",
    "source_present_missing_element_count",
    "recorded_zero_element_count",
    "recorded_negative_element_count",
    "supply_outcome_recorded_count",
    "supply_outcomes_complete",
    "negative_domestic_supply_flag",
]

register_columns = [
    column
    for column in preferred_register_columns
    if column in scoring_register_2024.columns
]

canonical_risk_register_2024 = (
    scoring_register_2024[
        register_columns
    ]
    .copy()
    .sort_values(
        [
            "eligible_for_2024_scoring",
            "predicted_shortage_probability",
            "Area",
            "Item Code",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

formal_warning_register_2024 = (
    canonical_risk_register_2024.loc[
        canonical_risk_register_2024[
            "predicted_shortage_warning"
        ]
        .fillna(0)
        .eq(1)
    ]
    .copy()
    .sort_values(
        "predicted_shortage_probability",
        ascending=False,
    )
    .reset_index(drop=True)
)

high_warning_register_2024 = (
    canonical_risk_register_2024.loc[
        canonical_risk_register_2024[
            "predicted_shortage_probability"
        ].ge(0.30)
    ]
    .copy()
    .sort_values(
        "predicted_shortage_probability",
        ascending=False,
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Create the exact 129-feature scoring table
# ------------------------------------------------------------

operational_features_2023 = (
    features_2023_all[
        locked_feature_names
    ]
    .copy()
)

assert len(operational_features_2023) == 344

assert operational_features_2023.columns.tolist() == (
    locked_feature_names
)

assert not np.isinf(
    operational_features_2023
    .select_dtypes(include=[np.number])
    .to_numpy()
).any()


# ------------------------------------------------------------
# 4. Create saved audit tables
# ------------------------------------------------------------

forward_scoring_decision_record = pd.DataFrame(
    {
        "Decision component": [
            "Predictor year",
            "Target year",
            "Available country-commodity grid",
            "Scored observations",
            "Unscored observations",
            "Current calorie eligibility minimum",
            "Probability threshold",
            "Watch-or-higher observations",
            "Formal warnings",
            "High warnings",
            "Usual-support observations",
            "Heightened-review observations",
            "Limited-evidence observations",
            "Headline-ranking observations",
            "Formal warnings excluded from headline ranking",
            "Risk-band PSI against locked test",
            "Observed 2024 outcomes used",
            "Threshold changed after scoring",
            "Intended interpretation",
        ],
        "Decision": [
            2023,
            2024,
            344,
            int(eligible_mask.sum()),
            int((~eligible_mask).sum()),
            minimum_current_kcal,
            probability_threshold,
            int(watch_or_higher_mask.sum()),
            int(formal_warning_mask.sum()),
            int(high_warning_mask.sum()),
            int(
                (
                    scoring_register_2024[
                        "evidence_review_level"
                    ]
                    == "Within usual historical support"
                ).sum()
            ),
            int(
                (
                    scoring_register_2024[
                        "evidence_review_level"
                    ]
                    == "Heightened evidence review"
                ).sum()
            ),
            int(limited_evidence_mask.sum()),
            int(
                scoring_register_2024[
                    "eligible_for_headline_ranking"
                ].sum()
            ),
            int(
                (
                    formal_warning_mask
                    & limited_evidence_mask
                ).sum()
            ),
            risk_band_psi,
            False,
            False,
            (
                "Model-generated early-warning screening "
                "estimate requiring human review"
            ),
        ],
    }
)


# ------------------------------------------------------------
# 5. Define output locations
# ------------------------------------------------------------

analysis_output_directory = (
    project_directory
    / "outputs"
    / "africa_first"
)

analysis_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

feature_output_path = (
    processed_data_directory
    / "africa_operational_shortage_features_2023.parquet"
)

risk_register_parquet_path = (
    analysis_output_directory
    / "africa_2024_shortage_risk_register.parquet"
)

risk_register_csv_path = (
    analysis_output_directory
    / "africa_2024_shortage_risk_register.csv"
)

formal_warning_path = (
    analysis_output_directory
    / "africa_2024_formal_warning_register.csv"
)

high_warning_path = (
    analysis_output_directory
    / "africa_2024_high_warning_register.csv"
)

eligibility_summary_path = (
    analysis_output_directory
    / "africa_2024_scoring_eligibility_summary.csv"
)

risk_band_summary_path = (
    analysis_output_directory
    / "africa_2024_risk_band_summary.csv"
)

distribution_comparison_path = (
    analysis_output_directory
    / "africa_2024_locked_test_distribution_comparison.csv"
)

evidence_support_path = (
    analysis_output_directory
    / "africa_2024_evidence_support_summary.csv"
)

decision_record_path = (
    analysis_output_directory
    / "africa_2024_scoring_decision_record.csv"
)

scoring_manifest_path = (
    analysis_output_directory
    / "africa_2024_scoring_manifest.json"
)


# ------------------------------------------------------------
# 6. Save the feature and analytical outputs
# ------------------------------------------------------------

operational_features_2023.to_parquet(
    feature_output_path,
    index=False,
)

canonical_risk_register_2024.to_parquet(
    risk_register_parquet_path,
    index=False,
)

canonical_risk_register_2024.to_csv(
    risk_register_csv_path,
    index=False,
)

formal_warning_register_2024.to_csv(
    formal_warning_path,
    index=False,
)

high_warning_register_2024.to_csv(
    high_warning_path,
    index=False,
)

eligibility_summary.to_csv(
    eligibility_summary_path,
    index=False,
)

risk_band_summary_2024.to_csv(
    risk_band_summary_path,
    index=False,
)

risk_distribution_comparison.to_csv(
    distribution_comparison_path,
    index=False,
)

evidence_support_summary.to_csv(
    evidence_support_path,
    index=False,
)

forward_scoring_decision_record.to_csv(
    decision_record_path,
    index=False,
)


# ------------------------------------------------------------
# 7. Record exact file fingerprints
# ------------------------------------------------------------

saved_data_paths = [
    feature_output_path,
    risk_register_parquet_path,
    risk_register_csv_path,
    formal_warning_path,
    high_warning_path,
    eligibility_summary_path,
    risk_band_summary_path,
    distribution_comparison_path,
    evidence_support_path,
    decision_record_path,
]

saved_file_signatures = {
    path.name: calculate_sha256(path)
    for path in saved_data_paths
}


# ------------------------------------------------------------
# 8. Create the scoring manifest
# ------------------------------------------------------------

scoring_manifest = {
    "project":
        "Africa-first food-supply shortage predictor",

    "output_type":
        "Prospective early-warning risk register",

    "predictor_year":
        2023,

    "target_year":
        2024,

    "important_interpretation": (
        "These are model-generated early-warning "
        "estimates, not observed 2024 shortage events."
    ),

    "source_grid": {
        "observations": 344,
        "countries": 43,
        "commodities": 8,
    },

    "prospective_eligibility": {
        "rule": (
            "The 2023 food-supply calorie value must be "
            "recorded and at least 5 kcal/person/day."
        ),
        "scored_observations":
            int(eligible_mask.sum()),
        "unscored_observations":
            int((~eligible_mask).sum()),
        "minimum_current_kcal_capita_day":
            float(minimum_current_kcal),
        "future_2024_outcome_required":
            False,
    },

    "model": {
        "file": operational_model_path.name,
        "file_signature_sha256":
            current_model_signature,
        "raw_features":
            len(locked_feature_names),
        "probability_threshold":
            float(probability_threshold),
        "model_design_changed":
            False,
        "threshold_changed":
            False,
    },

    "feature_reconstruction": {
        "historical_cells_compared":
            418992,
        "historical_mismatched_cells":
            0,
        "percentage_change_denominator":
            "Absolute previous value",
        "rolling_window_years":
            3,
        "rolling_window_includes_current_year":
            True,
        "rolling_minimum_recorded_years":
            2,
        "rolling_standard_deviation_ddof":
            0,
        "per_person_quantity_multiplier":
            1000000,
        "population_logarithm":
            "Natural logarithm plus one",
        "eligible_negative_domestic_supply_rows":
            0,
    },

    "risk_results": {
        "watch_or_higher":
            int(watch_or_higher_mask.sum()),
        "formal_warnings":
            int(formal_warning_mask.sum()),
        "high_warnings":
            int(high_warning_mask.sum()),
        "mean_probability":
            float(forward_probabilities.mean()),
        "median_probability":
            float(forward_probabilities.median()),
        "maximum_probability":
            float(forward_probabilities.max()),
    },

    "distribution_audit": {
        "locked_test_observations":
            int(len(locked_test_probabilities)),
        "forward_scored_observations":
            int(len(forward_probabilities)),
        "locked_test_warning_share":
            float(
                locked_test_probabilities
                .ge(probability_threshold)
                .mean()
            ),
        "forward_warning_share":
            float(
                forward_probabilities
                .ge(probability_threshold)
                .mean()
            ),
        "risk_band_population_stability_index":
            float(risk_band_psi),
        "psi_interpretation": (
            "Descriptive distribution-shift diagnostic; "
            "not predictive accuracy"
        ),
    },

    "evidence_review": {
        "historical_missing_feature_median":
            float(
                historical_missing_feature_count
                .median()
            ),
        "historical_missing_feature_q75":
            historical_missing_q75,
        "historical_missing_feature_q95":
            historical_missing_q95,
        "within_usual_support":
            195,
        "heightened_review":
            53,
        "limited_evidence":
            1,
    },

    "headline_ranking_policy": {
        "eligible_observations":
            int(
                scoring_register_2024[
                    "eligible_for_headline_ranking"
                ].sum()
            ),
        "limited_evidence_excluded":
            True,
        "excluded_warning_retained_in_full_register":
            True,
        "single_raw_warning_count_is_not_a_fair_ranking":
            True,
    },

    "reported_model_evaluation": {
        "must_use_locked_test_results":
            True,
        "operational_refit_scores_are_not_new_test_results":
            True,
    },

    "saved_file_signatures_sha256":
        saved_file_signatures,

    "created_utc":
        datetime.now(timezone.utc).isoformat(),
}

with open(
    scoring_manifest_path,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        scoring_manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 9. Reload and validate all principal evidence
# ------------------------------------------------------------

reloaded_features_2023 = pd.read_parquet(
    feature_output_path
)

reloaded_risk_register = pd.read_parquet(
    risk_register_parquet_path
)

reloaded_formal_warnings = pd.read_csv(
    formal_warning_path
)

reloaded_high_warnings = pd.read_csv(
    high_warning_path
)

with open(
    scoring_manifest_path,
    "r",
    encoding="utf-8",
) as manifest_file:
    reloaded_scoring_manifest = json.load(
        manifest_file
    )

assert len(reloaded_features_2023) == 344
assert len(reloaded_features_2023.columns) == 129

assert reloaded_features_2023.columns.tolist() == (
    locked_feature_names
)

assert len(reloaded_risk_register) == 344

assert reloaded_risk_register[
    "eligible_for_2024_scoring"
].sum() == 249

assert reloaded_risk_register[
    "predicted_shortage_probability"
].notna().sum() == 249

assert (
    reloaded_risk_register[
        "predicted_shortage_warning"
    ]
    .fillna(0)
    .sum()
    == 77
)

assert len(reloaded_formal_warnings) == 77
assert len(reloaded_high_warnings) == 12

assert reloaded_risk_register[
    "eligible_for_headline_ranking"
].sum() == 248

assert (
    reloaded_scoring_manifest[
        "risk_results"
    ]["formal_warnings"]
    == 77
)

assert (
    reloaded_scoring_manifest[
        "risk_results"
    ]["high_warnings"]
    == 12
)

for saved_path in saved_data_paths:

    assert calculate_sha256(saved_path) == (
        reloaded_scoring_manifest[
            "saved_file_signatures_sha256"
        ][saved_path.name]
    )


# ------------------------------------------------------------
# 10. Display saved-output summary
# ------------------------------------------------------------

saved_output_summary = pd.DataFrame(
    [
        {
            "Output": path.name,
            "Rows": (
                len(pd.read_parquet(path))
                if path.suffix == ".parquet"
                else (
                    len(pd.read_csv(path))
                    if path.suffix == ".csv"
                    else np.nan
                )
            ),
            "File size MB":
                path.stat().st_size / (1024 ** 2),
        }
        for path in saved_data_paths
    ]
    + [
        {
            "Output":
                scoring_manifest_path.name,
            "Rows": np.nan,
            "File size MB":
                scoring_manifest_path.stat().st_size
                / (1024 ** 2),
        }
    ]
)

print("Saved forward-scoring evidence:")
display(
    saved_output_summary.style.format(
        {
            "Rows": "{:,.0f}",
            "File size MB": "{:.4f}",
        },
        na_rep="—",
    )
)

print("\nForward-scoring decision record:")
display(forward_scoring_decision_record)

print(
    "\nThe verified 2023 feature table and 2024 "
    "early-warning register were saved, reloaded and "
    "validated successfully.\n"
    "All model probabilities and the 0.13 threshold "
    "remain unchanged.\n"
    "The estimates remain model-generated warnings, "
    "not observed 2024 shortage events."
)

Saved forward-scoring evidence:


,Output,Rows,File size MB
0,africa_operational_shortage_features_2023.parquet,344,0.2082
1,africa_2024_shortage_risk_register.parquet,344,0.0348
2,africa_2024_shortage_risk_register.csv,344,0.1142
3,africa_2024_formal_warning_register.csv,77,0.0286
4,africa_2024_high_warning_register.csv,12,0.0052
5,africa_2024_scoring_eligibility_summary.csv,3,0.0002
6,africa_2024_risk_band_summary.csv,4,0.0004
7,africa_2024_locked_test_distribution_comparison.csv,4,0.0005
8,africa_2024_evidence_support_summary.csv,3,0.0005
9,africa_2024_scoring_decision_record.csv,19,0.0007



Forward-scoring decision record:


,Decision component,Decision
0,Predictor year,2023
1,Target year,2024
2,Available country-commodity grid,344
3,Scored observations,249
4,Unscored observations,95
5,Current calorie eligibility minimum,5.0
6,Probability threshold,0.13
7,Watch-or-higher observations,156
8,Formal warnings,77
9,High warnings,12



The verified 2023 feature table and 2024 early-warning register were saved, reloaded and validated successfully.
All model probabilities and the 0.13 threshold remain unchanged.
The estimates remain model-generated warnings, not observed 2024 shortage events.


In [11]:
# ============================================================
# CORRECTION: WILSON SCORE FUNCTION
# ============================================================

def wilson_score_interval(
    events,
    observations,
    z_value=1.959963984540054,
):

    events = np.asarray(
        events,
        dtype=float,
    )

    observations = np.asarray(
        observations,
        dtype=float,
    )

    lower = np.full(
        observations.shape,
        np.nan,
        dtype=float,
    )

    upper = np.full(
        observations.shape,
        np.nan,
        dtype=float,
    )

    valid = observations > 0

    proportion = (
        events[valid]
        / observations[valid]
    )

    z_squared = z_value ** 2

    denominator = (
        1
        + z_squared
        / observations[valid]
    )

    centre = (
        proportion
        + z_squared
        / (
            2
            * observations[valid]
        )
    ) / denominator

    margin = (
        z_value
        * np.sqrt(
            (
                proportion
                * (1 - proportion)
                / observations[valid]
            )
            + (
                z_squared
                / (
                    4
                    * observations[valid] ** 2
                )
            )
        )
        / denominator
    )

    lower[valid] = np.maximum(
        0,
        centre - margin,
    )

    upper[valid] = np.minimum(
        1,
        centre + margin,
    )

    return lower, upper


# Brief technical check
test_lower, test_upper = wilson_score_interval(
    events=[10, 20],
    observations=[100, 100],
)

assert np.isfinite(test_lower).all()
assert np.isfinite(test_upper).all()
assert (test_lower <= test_upper).all()

print(
    "Wilson score function corrected successfully. "
    "Rerun the entire Step 7 cell."
)

Wilson score function corrected successfully. Rerun the entire Step 7 cell.


In [13]:
# ============================================================
# NOTEBOOK 5 — STEP 7
# BUILD THE HISTORICAL COUNTRY EVIDENCE TABLE
#
# Historical observed events and model-generated 2024 warnings
# remain explicitly separate.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Load the canonical supervised historical panel
# ------------------------------------------------------------

supervised_history_path = (
    processed_data_directory
    / "africa_supervised_shortage_panel_2010_2022.parquet"
)

assert supervised_history_path.exists()

historical_outcomes = pd.read_parquet(
    supervised_history_path
)

historical_outcomes["Area"] = (
    historical_outcomes["Area"].astype(str)
)

historical_outcomes["Item Code"] = (
    pd.to_numeric(
        historical_outcomes["Item Code"],
        errors="raise",
    )
    .astype("int64")
    .astype(str)
)

historical_outcomes["Year"] = (
    pd.to_numeric(
        historical_outcomes["Year"],
        errors="raise",
    )
    .astype("int16")
)

historical_outcomes["target_year"] = (
    pd.to_numeric(
        historical_outcomes["target_year"],
        errors="raise",
    )
    .astype("int16")
)

historical_outcomes[
    "shortage_next_year"
] = (
    historical_outcomes[
        "shortage_next_year"
    ]
    .astype("int8")
)

historical_key_columns = [
    "Area",
    "Item Code",
    "Year",
]

assert len(historical_outcomes) == 3248
assert historical_outcomes[
    "shortage_next_year"
].sum() == 334

assert historical_outcomes["Year"].min() == 2010
assert historical_outcomes["Year"].max() == 2022

assert historical_outcomes[
    "target_year"
].min() == 2011

assert historical_outcomes[
    "target_year"
].max() == 2023

assert not historical_outcomes.duplicated(
    historical_key_columns
).any()


# ------------------------------------------------------------
# 2. Cross-check against the model-ready historical table
# ------------------------------------------------------------

model_ready_target_check = (
    model_ready_history[
        historical_key_columns
        + ["target_year", "shortage_next_year"]
    ]
    .copy()
)

model_ready_target_check["Area"] = (
    model_ready_target_check["Area"].astype(str)
)

model_ready_target_check["Item Code"] = (
    pd.to_numeric(
        model_ready_target_check["Item Code"],
        errors="raise",
    )
    .astype("int64")
    .astype(str)
)

historical_target_comparison = (
    historical_outcomes[
        historical_key_columns
        + ["target_year", "shortage_next_year"]
    ]
    .merge(
        model_ready_target_check,
        on=historical_key_columns,
        how="outer",
        validate="one_to_one",
        suffixes=(
            "_supervised",
            "_model_ready",
        ),
        indicator=True,
    )
)

assert historical_target_comparison[
    "_merge"
].eq("both").all()

assert (
    historical_target_comparison[
        "target_year_supervised"
    ]
    ==
    historical_target_comparison[
        "target_year_model_ready"
    ]
).all()

assert (
    historical_target_comparison[
        "shortage_next_year_supervised"
    ]
    ==
    historical_target_comparison[
        "shortage_next_year_model_ready"
    ]
).all()


# ------------------------------------------------------------
# 3. Create the observed historical event ledger
# ------------------------------------------------------------

historical_outcomes[
    "outcome_type"
] = np.where(
    historical_outcomes[
        "shortage_next_year"
    ].eq(1),
    "Observed shortage event",
    "Observed non-shortage outcome",
)

historical_outcomes[
    "predictor_year"
] = historical_outcomes["Year"]

historical_outcomes[
    "event_year"
] = historical_outcomes["target_year"]

event_ledger_candidates = [
    "Area Code",
    "Area Code (M49)",
    "M49 Code",
    "Area",
    "Item Code (FBS)",
    "Item Code",
    "Item",
    "predictor_year",
    "event_year",
    "shortage_next_year",
    "outcome_type",
    "food_supply_kcal_cap_day",
    "next_food_supply_kcal_cap_day",
    "food_supply_kcal_cap_day_next_year",
    "absolute_kcal_decline",
    "percentage_kcal_decline",
    "kcal_absolute_change_next_year",
    "kcal_percentage_change_next_year",
    "production_status",
    "recorded_element_count",
    "source_absent_element_count",
    "source_present_missing_element_count",
    "supply_outcome_recorded_count",
]

available_event_ledger_columns = [
    column
    for column in event_ledger_candidates
    if column in historical_outcomes.columns
]

historical_event_ledger = (
    historical_outcomes.loc[
        historical_outcomes[
            "shortage_next_year"
        ].eq(1),
        available_event_ledger_columns,
    ]
    .copy()
    .sort_values(
        [
            "event_year",
            "Area",
            "Item Code",
        ]
    )
    .reset_index(drop=True)
)

assert len(historical_event_ledger) == 334


# ------------------------------------------------------------
# 4. Define a denominator-aware Wilson score interval
# ------------------------------------------------------------
# This provides a conservative rate boundary that responds to
# the number of eligible observations.
#
# Because observations repeat across commodities and years,
# these bounds are ranking aids rather than formal independent-
# sample confidence intervals.

def wilson_score_interval(
    events,
    observations,
    z_value=1.959963984540054,
):

    events = np.asarray(
        events,
        dtype=float,
    )

    observations = np.asarray(
        observations,
        dtype=float,
    )

    lower = np.full(
        len(observations),
        np.nan,
        dtype=float,
    )

    upper = np.full(
        len(observations),
        np.nan,
        dtype=float,
    )

    valid = observations > 0

    proportion = (
        events[valid]
        / observations[valid]
    )

    z_squared = z_value ** 2

    denominator = (
        1
        + z_squared
        / observations[valid]
    )

    centre = (
        proportion
        + z_squared
        / (
            2
            * observations[valid]
        )
    ) / denominator

    margin = (
        z_value
        * np.sqrt(
            (
                proportion
                * (1 - proportion)
                / observations[valid]
            )
            + (
                z_squared
                / (
                    4
                    * observations[valid] ** 2
                )
            )
        )
        / denominator
    )

    lower[valid] = np.maximum(
        0,
        centre - margin,
    )

    upper[valid] = np.minimum(
        1,
        centre + margin,
    )

    return lower, upper


# ------------------------------------------------------------
# 5. Create the full-period country summary
# ------------------------------------------------------------

country_full_history = (
    historical_outcomes
    .groupby(
        "Area",
        observed=True,
    )
    .agg(
        Historical_eligible_observations=(
            "shortage_next_year",
            "size",
        ),
        Historical_shortage_events=(
            "shortage_next_year",
            "sum",
        ),
        Historical_commodities_assessed=(
            "Item Code",
            "nunique",
        ),
        Historical_outcome_years_assessed=(
            "target_year",
            "nunique",
        ),
        First_outcome_year=(
            "target_year",
            "min",
        ),
        Last_outcome_year=(
            "target_year",
            "max",
        ),
    )
    .reset_index()
)

country_full_history[
    "Historical_event_rate"
] = (
    country_full_history[
        "Historical_shortage_events"
    ]
    / country_full_history[
        "Historical_eligible_observations"
    ]
)

(
    country_full_history[
        "Historical_rate_lower_bound"
    ],
    country_full_history[
        "Historical_rate_upper_bound"
    ],
) = wilson_score_interval(
    country_full_history[
        "Historical_shortage_events"
    ],
    country_full_history[
        "Historical_eligible_observations"
    ],
)

maximum_historical_observations = (
    8 * 13
)

country_full_history[
    "Historical_eligibility_share"
] = (
    country_full_history[
        "Historical_eligible_observations"
    ]
    / maximum_historical_observations
)


# ------------------------------------------------------------
# 6. Create the recent-period summary: 2019–2023
# ------------------------------------------------------------

recent_outcome_year_start = 2019
recent_outcome_year_end = 2023

recent_historical_outcomes = (
    historical_outcomes.loc[
        historical_outcomes[
            "target_year"
        ].between(
            recent_outcome_year_start,
            recent_outcome_year_end,
        )
    ]
    .copy()
)

country_recent_history = (
    recent_historical_outcomes
    .groupby(
        "Area",
        observed=True,
    )
    .agg(
        Recent_eligible_observations=(
            "shortage_next_year",
            "size",
        ),
        Recent_shortage_events=(
            "shortage_next_year",
            "sum",
        ),
        Recent_commodities_assessed=(
            "Item Code",
            "nunique",
        ),
        Recent_outcome_years_assessed=(
            "target_year",
            "nunique",
        ),
    )
    .reset_index()
)

country_recent_history[
    "Recent_event_rate"
] = (
    country_recent_history[
        "Recent_shortage_events"
    ]
    / country_recent_history[
        "Recent_eligible_observations"
    ]
)

(
    country_recent_history[
        "Recent_rate_lower_bound"
    ],
    country_recent_history[
        "Recent_rate_upper_bound"
    ],
) = wilson_score_interval(
    country_recent_history[
        "Recent_shortage_events"
    ],
    country_recent_history[
        "Recent_eligible_observations"
    ],
)

maximum_recent_observations = (
    8 * 5
)

country_recent_history[
    "Recent_eligibility_share"
] = (
    country_recent_history[
        "Recent_eligible_observations"
    ]
    / maximum_recent_observations
)


# ------------------------------------------------------------
# 7. Measure shortage breadth and recurrence
# ------------------------------------------------------------

country_commodity_history = (
    historical_outcomes
    .groupby(
        ["Area", "Item Code", "Item"],
        observed=True,
    )
    .agg(
        Eligible_observations=(
            "shortage_next_year",
            "size",
        ),
        Shortage_events=(
            "shortage_next_year",
            "sum",
        ),
        First_outcome_year=(
            "target_year",
            "min",
        ),
        Last_outcome_year=(
            "target_year",
            "max",
        ),
    )
    .reset_index()
)

country_commodity_history[
    "Commodity_event_rate"
] = (
    country_commodity_history[
        "Shortage_events"
    ]
    / country_commodity_history[
        "Eligible_observations"
    ]
)

country_commodity_history[
    "Commodity_had_shortage"
] = (
    country_commodity_history[
        "Shortage_events"
    ].gt(0)
)

country_commodity_history[
    "Commodity_had_recurrent_shortages"
] = (
    country_commodity_history[
        "Shortage_events"
    ].ge(2)
)

country_breadth_summary = (
    country_commodity_history
    .groupby(
        "Area",
        observed=True,
    )
    .agg(
        Eligible_commodity_pairs=(
            "Item Code",
            "size",
        ),
        Commodities_with_shortages=(
            "Commodity_had_shortage",
            "sum",
        ),
        Commodities_with_recurrent_shortages=(
            "Commodity_had_recurrent_shortages",
            "sum",
        ),
        Maximum_events_in_one_commodity=(
            "Shortage_events",
            "max",
        ),
    )
    .reset_index()
)

country_breadth_summary[
    "Historical_shortage_breadth_rate"
] = (
    country_breadth_summary[
        "Commodities_with_shortages"
    ]
    / country_breadth_summary[
        "Eligible_commodity_pairs"
    ]
)


# ------------------------------------------------------------
# 8. Identify the most recent observed event by country
# ------------------------------------------------------------

country_last_event = (
    historical_event_ledger
    .groupby(
        "Area",
        observed=True,
    )
    .agg(
        Most_recent_shortage_year=(
            "event_year",
            "max",
        )
    )
    .reset_index()
)

country_last_event[
    "Years_from_last_event_to_2023"
] = (
    2023
    - country_last_event[
        "Most_recent_shortage_year"
    ]
)


# ------------------------------------------------------------
# 9. Construct the evidence-qualified forward summary
# ------------------------------------------------------------

forward_ranking_source = (
    scoring_register_2024.loc[
        scoring_register_2024[
            "eligible_for_headline_ranking"
        ]
    ]
    .copy()
)

country_forward_summary = (
    forward_ranking_source
    .groupby(
        "Area",
        observed=True,
    )
    .agg(
        Forward_scored_commodities=(
            "Item Code",
            "size",
        ),
        Forward_watch_or_higher=(
            "predicted_shortage_probability",
            lambda values: int(
                values.ge(0.05).sum()
            ),
        ),
        Forward_formal_warnings=(
            "predicted_shortage_warning",
            "sum",
        ),
        Forward_high_warnings=(
            "predicted_shortage_probability",
            lambda values: int(
                values.ge(0.30).sum()
            ),
        ),
        Forward_average_probability=(
            "predicted_shortage_probability",
            "mean",
        ),
        Forward_maximum_probability=(
            "predicted_shortage_probability",
            "max",
        ),
    )
    .reset_index()
)

country_forward_summary[
    "Forward_warning_share"
] = (
    country_forward_summary[
        "Forward_formal_warnings"
    ]
    / country_forward_summary[
        "Forward_scored_commodities"
    ]
)

country_forward_summary[
    "Forward_watch_share"
] = (
    country_forward_summary[
        "Forward_watch_or_higher"
    ]
    / country_forward_summary[
        "Forward_scored_commodities"
    ]
)


# ------------------------------------------------------------
# 10. Combine the three evidence dimensions
# ------------------------------------------------------------

country_evidence_table = (
    country_full_history
    .merge(
        country_recent_history,
        on="Area",
        how="left",
        validate="one_to_one",
    )
    .merge(
        country_breadth_summary,
        on="Area",
        how="left",
        validate="one_to_one",
    )
    .merge(
        country_last_event,
        on="Area",
        how="left",
        validate="one_to_one",
    )
    .merge(
        country_forward_summary,
        on="Area",
        how="left",
        validate="one_to_one",
    )
)

assert len(country_evidence_table) == 43
assert country_evidence_table["Area"].nunique() == 43


# ------------------------------------------------------------
# 11. Create three separate, transparent ranks
# ------------------------------------------------------------
# Historical and recent ranks use the conservative lower rate
# boundary. Forward priority uses average probability across
# evidence-qualified, currently relevant commodities.

country_evidence_table[
    "Historical_burden_rank"
] = (
    country_evidence_table[
        "Historical_rate_lower_bound"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype("int16")
)

country_evidence_table[
    "Recent_burden_rank"
] = (
    country_evidence_table[
        "Recent_rate_lower_bound"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype("int16")
)

country_evidence_table[
    "Forward_screening_priority_rank"
] = (
    country_evidence_table[
        "Forward_average_probability"
    ]
    .rank(
        method="min",
        ascending=False,
        na_option="bottom",
    )
    .astype("int16")
)


# ------------------------------------------------------------
# 12. Create continental historical summaries
# ------------------------------------------------------------

continental_historical_summary = pd.DataFrame(
    {
        "Measure": [
            "Historical outcome observations",
            "Observed shortage events",
            "Observed non-shortage outcomes",
            "Historical event rate",
            "Countries",
            "Commodities",
            "Outcome year start",
            "Outcome year end",
            "Recent-period start",
            "Recent-period end",
            "Recent observations",
            "Recent shortage events",
            "Recent event rate",
        ],
        "Value": [
            len(historical_outcomes),
            int(
                historical_outcomes[
                    "shortage_next_year"
                ].sum()
            ),
            int(
                historical_outcomes[
                    "shortage_next_year"
                ].eq(0).sum()
            ),
            historical_outcomes[
                "shortage_next_year"
            ].mean(),
            historical_outcomes[
                "Area"
            ].nunique(),
            historical_outcomes[
                "Item Code"
            ].nunique(),
            historical_outcomes[
                "target_year"
            ].min(),
            historical_outcomes[
                "target_year"
            ].max(),
            recent_outcome_year_start,
            recent_outcome_year_end,
            len(recent_historical_outcomes),
            int(
                recent_historical_outcomes[
                    "shortage_next_year"
                ].sum()
            ),
            recent_historical_outcomes[
                "shortage_next_year"
            ].mean(),
        ],
    }
)

continental_annual_history = (
    historical_outcomes
    .groupby(
        "target_year",
        observed=True,
    )
    .agg(
        Eligible_observations=(
            "shortage_next_year",
            "size",
        ),
        Shortage_events=(
            "shortage_next_year",
            "sum",
        ),
        Countries_assessed=(
            "Area",
            "nunique",
        ),
        Commodities_assessed=(
            "Item Code",
            "nunique",
        ),
    )
    .reset_index()
    .rename(
        columns={
            "target_year": "Outcome year"
        }
    )
)

continental_annual_history[
    "Event_rate"
] = (
    continental_annual_history[
        "Shortage_events"
    ]
    / continental_annual_history[
        "Eligible_observations"
    ]
)


# ------------------------------------------------------------
# 13. Display the historical evidence
# ------------------------------------------------------------

print("Continental historical summary:")
display(
    continental_historical_summary.style.format(
        {
            "Value": "{:,.4f}",
        }
    )
)

print("\nAnnual observed shortage history:")
display(
    continental_annual_history.style.format(
        {
            "Event_rate": "{:.2%}",
        }
    )
)

historical_ranking_view = (
    country_evidence_table[
        [
            "Historical_burden_rank",
            "Area",
            "Historical_eligible_observations",
            "Historical_shortage_events",
            "Historical_event_rate",
            "Historical_rate_lower_bound",
            "Historical_rate_upper_bound",
            "Historical_commodities_assessed",
            "Commodities_with_shortages",
            "Commodities_with_recurrent_shortages",
            "Most_recent_shortage_year",
        ]
    ]
    .sort_values(
        "Historical_burden_rank"
    )
)

print("\nFull historical country ranking:")
display(
    historical_ranking_view.style.format(
        {
            "Historical_event_rate": "{:.2%}",
            "Historical_rate_lower_bound": "{:.2%}",
            "Historical_rate_upper_bound": "{:.2%}",
            "Most_recent_shortage_year": "{:.0f}",
        },
        na_rep="No recorded event",
    )
)

recent_ranking_view = (
    country_evidence_table[
        [
            "Recent_burden_rank",
            "Area",
            "Recent_eligible_observations",
            "Recent_shortage_events",
            "Recent_event_rate",
            "Recent_rate_lower_bound",
            "Recent_rate_upper_bound",
            "Recent_commodities_assessed",
            "Historical_burden_rank",
        ]
    ]
    .sort_values(
        "Recent_burden_rank"
    )
)

print("\nRecent historical country ranking, 2019–2023:")
display(
    recent_ranking_view.style.format(
        {
            "Recent_event_rate": "{:.2%}",
            "Recent_rate_lower_bound": "{:.2%}",
            "Recent_rate_upper_bound": "{:.2%}",
        }
    )
)

forward_ranking_view = (
    country_evidence_table[
        [
            "Forward_screening_priority_rank",
            "Area",
            "Forward_scored_commodities",
            "Forward_formal_warnings",
            "Forward_high_warnings",
            "Forward_warning_share",
            "Forward_average_probability",
            "Forward_maximum_probability",
            "Historical_burden_rank",
            "Recent_burden_rank",
        ]
    ]
    .sort_values(
        "Forward_screening_priority_rank"
    )
)

print(
    "\nForward 2024 screening-priority ranking "
    "(model-generated, not observed):"
)
display(
    forward_ranking_view.style.format(
        {
            "Forward_warning_share": "{:.2%}",
            "Forward_average_probability": "{:.4f}",
            "Forward_maximum_probability": "{:.4f}",
        }
    )
)

print(
    "\nHistorical country evidence constructed in memory.\n"
    "The historical, recent and forward ranks remain "
    "separate and no combined country score has been created."
)

Continental historical summary:


,Measure,Value
0,Historical outcome observations,"3,248.0000"
1,Observed shortage events,334.0000
2,Observed non-shortage outcomes,"2,914.0000"
3,Historical event rate,0.1028
4,Countries,43.0000
5,Commodities,8.0000
6,Outcome year start,"2,011.0000"
7,Outcome year end,"2,023.0000"
8,Recent-period start,"2,019.0000"
9,Recent-period end,"2,023.0000"



Annual observed shortage history:


,Outcome year,Eligible_observations,Shortage_events,Countries_assessed,Commodities_assessed,Event_rate
0,2011,249,25,43,8,10.04%
1,2012,251,18,43,8,7.17%
2,2013,252,27,43,8,10.71%
3,2014,251,32,43,8,12.75%
4,2015,250,26,43,8,10.40%
5,2016,249,29,43,8,11.65%
6,2017,249,14,43,8,5.62%
7,2018,249,26,43,8,10.44%
8,2019,250,34,43,8,13.60%
9,2020,246,24,43,8,9.76%



Full historical country ranking:


...


Recent historical country ranking, 2019–2023:


,Recent_burden_rank,Area,Recent_eligible_observations,Recent_shortage_events,Recent_event_rate,Recent_rate_lower_bound,Recent_rate_upper_bound,Recent_commodities_assessed,Historical_burden_rank
39,1,Uganda,34,11,32.35%,19.13%,49.16%,7,1
30,2,Niger,35,10,28.57%,16.33%,45.05%,7,2
25,3,Mauritania,21,5,23.81%,10.63%,45.09%,5,6
36,4,Sierra Leone,35,7,20.00%,10.04%,35.89%,7,7
10,5,Djibouti,18,4,22.22%,9.00%,45.21%,5,15
29,6,Namibia,30,5,16.67%,7.34%,33.56%,6,20
35,7,Seychelles,23,4,17.39%,6.98%,37.14%,5,9
24,8,Malawi,32,5,15.62%,6.86%,31.75%,7,5
20,9,Lesotho,24,4,16.67%,6.68%,35.85%,5,4
21,10,Liberia,25,4,16.00%,6.40%,34.65%,5,22



Forward 2024 screening-priority ranking (model-generated, not observed):


,Forward_screening_priority_rank,Area,Forward_scored_commodities,Forward_formal_warnings,Forward_high_warnings,Forward_warning_share,Forward_average_probability,Forward_maximum_probability,Historical_burden_rank,Recent_burden_rank
24,1,Malawi,7,4,2,57.14%,0.2449,0.6106,5,8
35,2,Seychelles,4,3,1,75.00%,0.2338,0.3979,9,7
39,3,Uganda,7,5,1,71.43%,0.1962,0.3182,1,1
20,4,Lesotho,4,3,1,75.00%,0.1834,0.3235,4,9
26,5,Mauritius,4,2,1,50.00%,0.1756,0.4445,29,35
22,6,Libya,4,3,0,75.00%,0.1685,0.2193,29,14
15,7,Gambia,7,3,2,42.86%,0.1484,0.3597,3,11
32,8,Rwanda,7,2,1,28.57%,0.1430,0.5035,14,11
7,9,Congo,6,2,1,33.33%,0.1399,0.6195,31,37
30,10,Niger,7,3,0,42.86%,0.1368,0.1934,2,2



Historical country evidence constructed in memory.
The historical, recent and forward ranks remain separate and no combined country score has been created.


In [14]:
# ============================================================
# NOTEBOOK 5 — STEP 8
# FINALISE AND SAVE THE CONTINENTAL, COUNTRY AND COMMODITY
# ANALYTICAL EVIDENCE TABLES
# ============================================================

from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Add transparent alternative country ranks
# ------------------------------------------------------------

country_evidence_table[
    "Historical_raw_rate_rank"
] = (
    country_evidence_table[
        "Historical_event_rate"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype("int16")
)

country_evidence_table[
    "Recent_raw_rate_rank"
] = (
    country_evidence_table[
        "Recent_event_rate"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype("int16")
)

country_evidence_table[
    "Forward_peak_risk_rank"
] = (
    country_evidence_table[
        "Forward_maximum_probability"
    ]
    .rank(
        method="min",
        ascending=False,
        na_option="bottom",
    )
    .astype("int16")
)

country_evidence_table[
    "Forward_warning_share_rank"
] = (
    country_evidence_table[
        "Forward_warning_share"
    ]
    .rank(
        method="min",
        ascending=False,
        na_option="bottom",
    )
    .astype("int16")
)

country_evidence_table[
    "Historical_rank_adjustment_for_denominator"
] = (
    country_evidence_table[
        "Historical_burden_rank"
    ]
    - country_evidence_table[
        "Historical_raw_rate_rank"
    ]
)

country_evidence_table[
    "Recent_rank_adjustment_for_denominator"
] = (
    country_evidence_table[
        "Recent_burden_rank"
    ]
    - country_evidence_table[
        "Recent_raw_rate_rank"
    ]
)


# ------------------------------------------------------------
# 2. Create a country rank-comparison audit
# ------------------------------------------------------------

country_rank_columns = [
    "Historical_burden_rank",
    "Recent_burden_rank",
    "Forward_screening_priority_rank",
    "Forward_peak_risk_rank",
]

country_rank_correlation = (
    country_evidence_table[
        country_rank_columns
    ]
    .corr(method="spearman")
    .reset_index()
    .rename(
        columns={"index": "Rank"}
    )
)

country_rank_divergence = (
    country_evidence_table[
        [
            "Area",
            "Historical_burden_rank",
            "Recent_burden_rank",
            "Forward_screening_priority_rank",
            "Forward_peak_risk_rank",
            "Historical_event_rate",
            "Recent_event_rate",
            "Forward_average_probability",
            "Forward_maximum_probability",
        ]
    ]
    .copy()
)

country_rank_divergence[
    "Historical_to_forward_rank_change"
] = (
    country_rank_divergence[
        "Historical_burden_rank"
    ]
    - country_rank_divergence[
        "Forward_screening_priority_rank"
    ]
)

country_rank_divergence[
    "Absolute_historical_forward_difference"
] = (
    country_rank_divergence[
        "Historical_to_forward_rank_change"
    ].abs()
)

country_rank_divergence = (
    country_rank_divergence
    .sort_values(
        "Absolute_historical_forward_difference",
        ascending=False,
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Build full-period commodity evidence
# ------------------------------------------------------------

commodity_country_history = (
    historical_outcomes
    .groupby(
        ["Item Code", "Item", "Area"],
        observed=True,
    )
    .agg(
        Eligible_observations=(
            "shortage_next_year",
            "size",
        ),
        Shortage_events=(
            "shortage_next_year",
            "sum",
        ),
    )
    .reset_index()
)

commodity_country_history[
    "Country_had_shortage"
] = (
    commodity_country_history[
        "Shortage_events"
    ].gt(0)
)

commodity_country_history[
    "Country_had_recurrent_shortages"
] = (
    commodity_country_history[
        "Shortage_events"
    ].ge(2)
)

commodity_full_history = (
    historical_outcomes
    .groupby(
        ["Item Code", "Item"],
        observed=True,
    )
    .agg(
        Historical_eligible_observations=(
            "shortage_next_year",
            "size",
        ),
        Historical_shortage_events=(
            "shortage_next_year",
            "sum",
        ),
        Countries_assessed=(
            "Area",
            "nunique",
        ),
        Outcome_years_assessed=(
            "target_year",
            "nunique",
        ),
    )
    .reset_index()
)

commodity_breadth = (
    commodity_country_history
    .groupby(
        ["Item Code", "Item"],
        observed=True,
    )
    .agg(
        Countries_with_shortages=(
            "Country_had_shortage",
            "sum",
        ),
        Countries_with_recurrent_shortages=(
            "Country_had_recurrent_shortages",
            "sum",
        ),
    )
    .reset_index()
)

commodity_full_history = (
    commodity_full_history
    .merge(
        commodity_breadth,
        on=["Item Code", "Item"],
        how="left",
        validate="one_to_one",
    )
)

commodity_full_history[
    "Historical_event_rate"
] = (
    commodity_full_history[
        "Historical_shortage_events"
    ]
    / commodity_full_history[
        "Historical_eligible_observations"
    ]
)

(
    commodity_full_history[
        "Historical_rate_lower_bound"
    ],
    commodity_full_history[
        "Historical_rate_upper_bound"
    ],
) = wilson_score_interval(
    commodity_full_history[
        "Historical_shortage_events"
    ],
    commodity_full_history[
        "Historical_eligible_observations"
    ],
)


# ------------------------------------------------------------
# 4. Build recent commodity evidence
# ------------------------------------------------------------

commodity_recent_history = (
    recent_historical_outcomes
    .groupby(
        ["Item Code", "Item"],
        observed=True,
    )
    .agg(
        Recent_eligible_observations=(
            "shortage_next_year",
            "size",
        ),
        Recent_shortage_events=(
            "shortage_next_year",
            "sum",
        ),
        Recent_countries_assessed=(
            "Area",
            "nunique",
        ),
    )
    .reset_index()
)

commodity_recent_history[
    "Recent_event_rate"
] = (
    commodity_recent_history[
        "Recent_shortage_events"
    ]
    / commodity_recent_history[
        "Recent_eligible_observations"
    ]
)

(
    commodity_recent_history[
        "Recent_rate_lower_bound"
    ],
    commodity_recent_history[
        "Recent_rate_upper_bound"
    ],
) = wilson_score_interval(
    commodity_recent_history[
        "Recent_shortage_events"
    ],
    commodity_recent_history[
        "Recent_eligible_observations"
    ],
)


# ------------------------------------------------------------
# 5. Build evidence-qualified forward commodity evidence
# ------------------------------------------------------------

commodity_forward_summary = (
    forward_ranking_source
    .groupby(
        ["Item Code", "Item"],
        observed=True,
    )
    .agg(
        Forward_scored_countries=(
            "Area",
            "nunique",
        ),
        Forward_watch_or_higher=(
            "predicted_shortage_probability",
            lambda values: int(
                values.ge(0.05).sum()
            ),
        ),
        Forward_formal_warnings=(
            "predicted_shortage_warning",
            "sum",
        ),
        Forward_high_warnings=(
            "predicted_shortage_probability",
            lambda values: int(
                values.ge(0.30).sum()
            ),
        ),
        Forward_average_probability=(
            "predicted_shortage_probability",
            "mean",
        ),
        Forward_maximum_probability=(
            "predicted_shortage_probability",
            "max",
        ),
    )
    .reset_index()
)

commodity_forward_summary[
    "Forward_warning_share"
] = (
    commodity_forward_summary[
        "Forward_formal_warnings"
    ]
    / commodity_forward_summary[
        "Forward_scored_countries"
    ]
)


# ------------------------------------------------------------
# 6. Create the combined commodity evidence table
# ------------------------------------------------------------

commodity_evidence_table = (
    commodity_full_history
    .merge(
        commodity_recent_history,
        on=["Item Code", "Item"],
        how="outer",
        validate="one_to_one",
    )
    .merge(
        commodity_forward_summary,
        on=["Item Code", "Item"],
        how="outer",
        validate="one_to_one",
    )
)

assert len(commodity_evidence_table) == 8
assert commodity_evidence_table[
    "Item Code"
].nunique() == 8

commodity_evidence_table[
    "Historical_commodity_burden_rank"
] = (
    commodity_evidence_table[
        "Historical_rate_lower_bound"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype("int16")
)

commodity_evidence_table[
    "Recent_commodity_burden_rank"
] = (
    commodity_evidence_table[
        "Recent_rate_lower_bound"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype("int16")
)

commodity_evidence_table[
    "Forward_commodity_priority_rank"
] = (
    commodity_evidence_table[
        "Forward_average_probability"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype("int16")
)

commodity_evidence_table = (
    commodity_evidence_table
    .sort_values(
        "Historical_commodity_burden_rank"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Final cross-table integrity checks
# ------------------------------------------------------------

assert country_evidence_table[
    "Historical_shortage_events"
].sum() == 334

assert country_evidence_table[
    "Recent_shortage_events"
].sum() == 137

assert commodity_evidence_table[
    "Historical_shortage_events"
].sum() == 334

assert commodity_evidence_table[
    "Recent_shortage_events"
].sum() == 137

assert continental_annual_history[
    "Shortage_events"
].sum() == 334

assert len(historical_event_ledger) == 334

assert country_evidence_table["Area"].nunique() == 43

assert country_evidence_table[
    "Forward_formal_warnings"
].sum() == 76
# One limited-evidence formal warning is deliberately excluded
# from headline country and commodity ranking inputs.

assert commodity_evidence_table[
    "Forward_formal_warnings"
].sum() == 76


# ------------------------------------------------------------
# 8. Display ranking sensitivity
# ------------------------------------------------------------

print("Spearman agreement between separate country ranks:")
display(
    country_rank_correlation.style.format(
        precision=3
    )
)

print(
    "\nCountries with the largest difference between "
    "historical and forward portfolio ranks:"
)
display(
    country_rank_divergence.head(15).style.format(
        {
            "Historical_event_rate": "{:.2%}",
            "Recent_event_rate": "{:.2%}",
            "Forward_average_probability":
                "{:.4f}",
            "Forward_maximum_probability":
                "{:.4f}",
        }
    )
)

print("\nCombined commodity evidence:")
display(
    commodity_evidence_table.style.format(
        {
            "Historical_event_rate": "{:.2%}",
            "Historical_rate_lower_bound": "{:.2%}",
            "Historical_rate_upper_bound": "{:.2%}",
            "Recent_event_rate": "{:.2%}",
            "Recent_rate_lower_bound": "{:.2%}",
            "Recent_rate_upper_bound": "{:.2%}",
            "Forward_warning_share": "{:.2%}",
            "Forward_average_probability": "{:.4f}",
            "Forward_maximum_probability": "{:.4f}",
        }
    )
)


# ------------------------------------------------------------
# 9. Save canonical analytical evidence
# ------------------------------------------------------------

historical_outcome_path = (
    analysis_output_directory
    / "africa_historical_outcome_register_2011_2023.parquet"
)

historical_event_ledger_path = (
    analysis_output_directory
    / "africa_observed_shortage_event_ledger_2011_2023.csv"
)

country_commodity_history_path = (
    analysis_output_directory
    / "africa_country_commodity_history_2011_2023.csv"
)

country_evidence_path = (
    analysis_output_directory
    / "africa_country_evidence_2011_2024.csv"
)

commodity_evidence_path = (
    analysis_output_directory
    / "africa_commodity_evidence_2011_2024.csv"
)

annual_history_path = (
    analysis_output_directory
    / "africa_continental_annual_history_2011_2023.csv"
)

continental_summary_path = (
    analysis_output_directory
    / "africa_continental_historical_summary_2011_2023.csv"
)

rank_correlation_path = (
    analysis_output_directory
    / "africa_country_rank_correlation.csv"
)

ranking_methodology_path = (
    analysis_output_directory
    / "africa_country_ranking_methodology.json"
)

historical_outcomes.to_parquet(
    historical_outcome_path,
    index=False,
)

historical_event_ledger.to_csv(
    historical_event_ledger_path,
    index=False,
)

country_commodity_history.to_csv(
    country_commodity_history_path,
    index=False,
)

country_evidence_table.to_csv(
    country_evidence_path,
    index=False,
)

commodity_evidence_table.to_csv(
    commodity_evidence_path,
    index=False,
)

continental_annual_history.to_csv(
    annual_history_path,
    index=False,
)

continental_historical_summary.to_csv(
    continental_summary_path,
    index=False,
)

country_rank_correlation.to_csv(
    rank_correlation_path,
    index=False,
)


# ------------------------------------------------------------
# 10. Save the ranking methodology
# ------------------------------------------------------------

ranking_methodology = {
    "project":
        "Africa-first food-supply shortage predictor",

    "historical_evidence": {
        "outcome_years": "2011–2023",
        "observations": 3248,
        "observed_shortage_events": 334,
        "countries": 43,
        "commodities": 8,
        "event_definition": (
            "Next-year calorie availability fell by at "
            "least 15 percent and at least 5 "
            "kcal/person/day, where current availability "
            "was at least 5 kcal/person/day."
        ),
    },

    "recent_period": {
        "outcome_years": "2019–2023",
        "observations":
            int(len(recent_historical_outcomes)),
        "observed_shortage_events":
            int(
                recent_historical_outcomes[
                    "shortage_next_year"
                ].sum()
            ),
    },

    "historical_country_rank": {
        "name":
            "Historical burden rank",
        "primary_measure":
            "Wilson lower bound for the full-period event rate",
        "reason": (
            "Uses the observed shortage rate while accounting "
            "for unequal eligible observation denominators."
        ),
        "important_limitation": (
            "Repeated country-commodity-year observations are "
            "not fully independent; Wilson bounds are used as "
            "conservative ranking aids, not formal causal or "
            "survey confidence intervals."
        ),
    },

    "recent_country_rank": {
        "name":
            "Recent burden rank",
        "period":
            "2019–2023",
        "primary_measure":
            "Wilson lower bound for the recent event rate",
    },

    "forward_country_ranks": {
        "portfolio_rank": (
            "Average predicted 2024 probability across "
            "evidence-qualified scored commodities"
        ),
        "peak_risk_rank": (
            "Maximum predicted 2024 commodity probability"
        ),
        "warning_share": (
            "Formal warnings divided by evidence-qualified "
            "scored commodities"
        ),
        "important_interpretation": (
            "Forward ranks are model-generated screening "
            "priorities, not observed events or national "
            "food-insecurity probabilities."
        ),
    },

    "limited_evidence_policy": {
        "full_register_retains_limited_evidence":
            True,
        "headline_ranking_excludes_limited_evidence":
            True,
        "excluded_2024_observations":
            1,
        "excluded_formal_warnings":
            1,
    },

    "combined_rank": {
        "created": False,
        "reason": (
            "Combining observed historical events, recent "
            "events and model probabilities would require "
            "subjective weights and could conceal important "
            "commodity-specific signals."
        ),
    },

    "reporting_requirement": (
        "Every country report must present denominators, "
        "historical rates, recent evidence, commodity breadth, "
        "average forward risk, peak forward risk and evidence "
        "limitations separately."
    ),

    "created_utc":
        datetime.now(timezone.utc).isoformat(),
}

with open(
    ranking_methodology_path,
    "w",
    encoding="utf-8",
) as methodology_file:
    json.dump(
        ranking_methodology,
        methodology_file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 11. Reload and validate saved evidence
# ------------------------------------------------------------

reloaded_historical_outcomes = pd.read_parquet(
    historical_outcome_path
)

reloaded_event_ledger = pd.read_csv(
    historical_event_ledger_path
)

reloaded_country_evidence = pd.read_csv(
    country_evidence_path
)

reloaded_commodity_evidence = pd.read_csv(
    commodity_evidence_path
)

reloaded_annual_history = pd.read_csv(
    annual_history_path
)

with open(
    ranking_methodology_path,
    "r",
    encoding="utf-8",
) as methodology_file:
    reloaded_ranking_methodology = json.load(
        methodology_file
    )

assert len(reloaded_historical_outcomes) == 3248
assert reloaded_historical_outcomes[
    "shortage_next_year"
].sum() == 334

assert len(reloaded_event_ledger) == 334
assert len(reloaded_country_evidence) == 43
assert len(reloaded_commodity_evidence) == 8
assert len(reloaded_annual_history) == 13

assert reloaded_country_evidence[
    "Historical_shortage_events"
].sum() == 334

assert reloaded_commodity_evidence[
    "Historical_shortage_events"
].sum() == 334

assert (
    reloaded_ranking_methodology[
        "combined_rank"
    ]["created"]
    is False
)


# ------------------------------------------------------------
# 12. Display saved evidence
# ------------------------------------------------------------

historical_saved_outputs = pd.DataFrame(
    [
        {
            "Output": path.name,
            "Rows": (
                len(pd.read_parquet(path))
                if path.suffix == ".parquet"
                else (
                    len(pd.read_csv(path))
                    if path.suffix == ".csv"
                    else np.nan
                )
            ),
            "File size MB":
                path.stat().st_size / (1024 ** 2),
        }
        for path in [
            historical_outcome_path,
            historical_event_ledger_path,
            country_commodity_history_path,
            country_evidence_path,
            commodity_evidence_path,
            annual_history_path,
            continental_summary_path,
            rank_correlation_path,
        ]
    ]
    + [
        {
            "Output":
                ranking_methodology_path.name,
            "Rows": np.nan,
            "File size MB":
                ranking_methodology_path.stat().st_size
                / (1024 ** 2),
        }
    ]
)

print("\nSaved analytical evidence:")
display(
    historical_saved_outputs.style.format(
        {
            "Rows": "{:,.0f}",
            "File size MB": "{:.4f}",
        },
        na_rep="—",
    )
)

print(
    "\nThe historical event ledger, country evidence, "
    "commodity evidence and ranking methodology were "
    "saved and validated successfully.\n"
    "No combined country rank was created."
)

Spearman agreement between separate country ranks:


,Rank,Historical_burden_rank,Recent_burden_rank,Forward_screening_priority_rank,Forward_peak_risk_rank
0,Historical_burden_rank,1.000,0.796,0.644,0.479
1,Recent_burden_rank,0.796,1.000,0.575,0.405
2,Forward_screening_priority_rank,0.644,0.575,1.000,0.769
3,Forward_peak_risk_rank,0.479,0.405,0.769,1.000



Countries with the largest difference between historical and forward portfolio ranks:


,Area,Historical_burden_rank,Recent_burden_rank,Forward_screening_priority_rank,Forward_peak_risk_rank,Historical_event_rate,Recent_event_rate,Forward_average_probability,Forward_maximum_probability,Historical_to_forward_rank_change,Absolute_historical_forward_difference
0,Mauritius,29,35,5,4,7.69%,5.00%,0.1756,0.4445,24,24
1,Libya,29,14,6,24,7.69%,15.00%,0.1685,0.2193,23,23
2,Congo,31,37,9,1,6.41%,3.33%,0.1399,0.6195,22,22
3,Cabo Verde,18,28,39,39,10.94%,8.00%,0.0524,0.1187,-21,21
4,Mozambique,13,16,33,37,12.36%,11.76%,0.0711,0.1276,-20,20
5,Algeria,43,41,24,30,0.00%,0.00%,0.0984,0.1717,19,19
6,Comoros,40,30,22,32,2.56%,6.67%,0.1018,0.1670,18,18
7,South Africa,39,36,23,22,3.33%,4.35%,0.0993,0.2217,16,16
8,Burkina Faso,17,15,31,25,10.34%,12.50%,0.0789,0.2174,-14,14
9,Botswana,16,22,27,14,10.96%,11.11%,0.0962,0.2643,-11,11



Combined commodity evidence:


,Item Code,Item,Historical_eligible_observations,Historical_shortage_events,Countries_assessed,Outcome_years_assessed,Countries_with_shortages,Countries_with_recurrent_shortages,Historical_event_rate,Historical_rate_lower_bound,Historical_rate_upper_bound,Recent_eligible_observations,Recent_shortage_events,Recent_countries_assessed,Recent_event_rate,Recent_rate_lower_bound,Recent_rate_upper_bound,Forward_scored_countries,Forward_watch_or_higher,Forward_formal_warnings,Forward_high_warnings,Forward_average_probability,Forward_maximum_probability,Forward_warning_share,Historical_commodity_burden_rank,Recent_commodity_burden_rank,Forward_commodity_priority_rank
0,2518,Sorghum and products,333,59,27,13,20,18,17.72%,13.99%,22.18%,127,23,27,18.11%,12.38%,25.71%,26,21,9,1,0.1153,0.3235,34.62%,1,1,3
1,2552,Groundnuts,464,64,40,13,25,22,13.79%,10.95%,17.23%,181,27,39,14.92%,10.46%,20.83%,35,28,20,1,0.1584,0.5291,57.14%,2,2,1
2,2517,Millet and products,280,34,23,13,16,12,12.14%,8.82%,16.49%,106,15,23,14.15%,8.77%,22.04%,21,14,7,1,0.1110,0.3234,33.33%,3,3,4
3,2511,Wheat and products,559,57,43,13,24,12,10.20%,7.95%,12.98%,215,26,43,12.09%,8.39%,17.13%,43,29,16,5,0.1313,0.6106,37.21%,4,4,2
4,2514,Maize and products,528,51,42,13,22,14,9.66%,7.42%,12.48%,204,20,41,9.80%,6.44%,14.66%,41,20,8,4,0.0989,0.6195,19.51%,5,5,5
5,2807,Rice and products,559,42,43,13,23,14,7.51%,5.61%,10.00%,215,15,43,6.98%,4.27%,11.19%,43,29,10,0,0.0916,0.2514,23.26%,6,6,6
6,2532,Cassava and products,362,24,28,13,14,6,6.63%,4.50%,9.68%,138,10,28,7.25%,3.98%,12.83%,27,9,5,0,0.0619,0.2111,18.52%,7,7,7
7,2535,Yams,163,3,14,13,3,0,1.84%,0.63%,5.27%,62,1,13,1.61%,0.29%,8.59%,12,5,1,0,0.0588,0.2405,8.33%,8,8,8



Saved analytical evidence:


,Output,Rows,File size MB
0,africa_historical_outcome_register_2011_2023.parquet,"3,248",0.2655
1,africa_observed_shortage_event_ledger_2011_2023.csv,334,0.0423
2,africa_country_commodity_history_2011_2023.csv,260,0.0178
3,africa_country_evidence_2011_2024.csv,43,0.0130
4,africa_commodity_evidence_2011_2024.csv,8,0.0025
5,africa_continental_annual_history_2011_2023.csv,13,0.0005
6,africa_continental_historical_summary_2011_2023.csv,13,0.0004
7,africa_country_rank_correlation.csv,4,0.0004
8,africa_country_ranking_methodology.json,—,0.0022



The historical event ledger, country evidence, commodity evidence and ranking methodology were saved and validated successfully.
No combined country rank was created.


In [15]:
# ============================================================
# NOTEBOOK 5 — STEP 9
# CREATE THE ANALYTICAL REPORTING FRAMEWORK AND
# STANDARDISED REPORT EVIDENCE PACKETS
#
# This creates structured report inputs, not finished prose.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import re
import unicodedata
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Define the reporting architecture
# ------------------------------------------------------------

reporting_framework = {
    "project":
        "Africa food-supply shortage analytical reports",

    "report_streams": {
        "methodology_report": {
            "purpose": (
                "Explain and defend the data preparation, "
                "target construction, feature engineering, "
                "model evaluation and operational scoring."
            ),
            "status":
                "Integrated methodology report completed",
        },

        "continental_analytical_report": {
            "purpose": (
                "Describe continent-wide historical events, "
                "country comparisons, commodity patterns and "
                "model-generated forward warnings."
            ),
            "planned_sections": [
                "Executive overview",
                "Scope and definitions",
                "Continental historical trend, 2011–2023",
                "Historical country comparison",
                "Recent country comparison, 2019–2023",
                "Commodity-level historical analysis",
                "2024 model-generated warning landscape",
                "Persistent, emerging and commodity-specific signals",
                "Evidence quality and limitations",
                "Country profile directory",
            ],
        },

        "country_reports": {
            "number_of_reports": 43,
            "purpose": (
                "Explain each country's historical commodity "
                "experience, recent events and model-generated "
                "2024 warnings."
            ),
            "planned_sections": [
                "Country evidence summary",
                "Coverage and commodities assessed",
                "Historical shortage record, 2011–2023",
                "Recent experience, 2019–2023",
                "Commodity-by-commodity history",
                "2024 model-generated risk register",
                "Highest-priority commodity signals",
                "Relevant supply and utilisation trends",
                "Evidence quality and interpretive cautions",
            ],
        },

        "commodity_reports": {
            "number_of_reports": 8,
            "purpose": (
                "Explain each commodity's historical shortage "
                "pattern across countries and its forward "
                "warning distribution."
            ),
            "planned_sections": [
                "Commodity overview",
                "Historical continental burden",
                "Annual event pattern",
                "Countries most affected historically",
                "Recent country pattern",
                "2024 model-generated warning register",
                "Highest-priority country signals",
                "Data coverage and limitations",
            ],
        },
    },

    "mandatory_reporting_rules": [
        (
            "Observed historical events must always be "
            "distinguished from model-generated warnings."
        ),
        (
            "Every rate must be accompanied by its eligible "
            "observation denominator."
        ),
        (
            "Historical, recent, forward-average and "
            "forward-peak ranks must remain separate."
        ),
        (
            "No combined rank may be introduced without an "
            "explicitly justified weighting policy."
        ),
        (
            "Forward probabilities must not be interpreted as "
            "national famine or food-insecurity probabilities."
        ),
        (
            "Limited-evidence warnings must remain visible but "
            "must not enter headline rankings."
        ),
        (
            "A model warning must not be described as an "
            "event that definitely occurred or will occur."
        ),
        (
            "Associations in production, imports, exports or "
            "supply must not be presented as causal effects."
        ),
        (
            "Country comparisons must show both average risk "
            "and the highest individual commodity risk."
        ),
        (
            "Unscored commodities must be reported as "
            "unscored, not as zero risk."
        ),
    ],

    "created_utc":
        datetime.now(timezone.utc).isoformat(),
}


# ------------------------------------------------------------
# 2. Define report-input directories
# ------------------------------------------------------------

report_input_directory = (
    analysis_output_directory
    / "report_inputs"
)

country_packet_directory = (
    report_input_directory
    / "countries"
)

commodity_packet_directory = (
    report_input_directory
    / "commodities"
)

country_packet_directory.mkdir(
    parents=True,
    exist_ok=True,
)

commodity_packet_directory.mkdir(
    parents=True,
    exist_ok=True,
)

reporting_framework_path = (
    report_input_directory
    / "analytical_reporting_framework.json"
)

continental_packet_path = (
    report_input_directory
    / "continental_analytical_evidence.json"
)

packet_index_path = (
    report_input_directory
    / "analytical_report_packet_index.csv"
)


# ------------------------------------------------------------
# 3. JSON conversion helpers
# ------------------------------------------------------------

def dataframe_to_json_records(dataframe):

    return json.loads(
        dataframe.to_json(
            orient="records",
            date_format="iso",
        )
    )


def dataframe_first_record(dataframe):

    records = dataframe_to_json_records(
        dataframe
    )

    return records[0] if records else {}


def make_safe_filename(value):

    normalised = unicodedata.normalize(
        "NFKD",
        str(value),
    )

    ascii_value = normalised.encode(
        "ascii",
        "ignore",
    ).decode("ascii")

    safe_value = re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        ascii_value,
    )

    return safe_value.strip("_").lower()


# ------------------------------------------------------------
# 4. Prepare report-ready analytical trends
# ------------------------------------------------------------

report_analytical_panel = (
    analytical_panel.copy()
)

report_analytical_panel["Area"] = (
    report_analytical_panel[
        "Area"
    ].astype(str)
)

report_analytical_panel["Item Code"] = (
    pd.to_numeric(
        report_analytical_panel["Item Code"],
        errors="raise",
    )
    .astype("int64")
    .astype(str)
)

report_analytical_panel["Year"] = (
    pd.to_numeric(
        report_analytical_panel["Year"],
        errors="raise",
    )
    .astype("int16")
)

trend_column_candidates = [
    "Area",
    "Item Code",
    "Item",
    "Year",
    "Population",
    "production_1000t",
    "import_quantity_1000t",
    "export_quantity_1000t",
    "domestic_supply_1000t",
    "food_1000t",
    "food_supply_kcal_cap_day",
    "food_supply_quantity_kg_cap_yr",
    "losses_1000t",
    "stock_variation_1000t",
    "production_status",
    "recorded_element_count",
    "source_absent_element_count",
    "source_present_missing_element_count",
    "supply_outcome_recorded_count",
]

available_trend_columns = [
    column
    for column in trend_column_candidates
    if column in report_analytical_panel.columns
]


# ------------------------------------------------------------
# 5. Create country-year observed histories
# ------------------------------------------------------------

country_annual_history = (
    historical_outcomes
    .groupby(
        ["Area", "target_year"],
        observed=True,
    )
    .agg(
        Eligible_observations=(
            "shortage_next_year",
            "size",
        ),
        Shortage_events=(
            "shortage_next_year",
            "sum",
        ),
        Commodities_assessed=(
            "Item Code",
            "nunique",
        ),
    )
    .reset_index()
    .rename(
        columns={
            "target_year": "Outcome_year"
        }
    )
)

country_annual_history[
    "Event_rate"
] = (
    country_annual_history[
        "Shortage_events"
    ]
    / country_annual_history[
        "Eligible_observations"
    ]
)


# ------------------------------------------------------------
# 6. Create commodity-year observed histories
# ------------------------------------------------------------

commodity_annual_history = (
    historical_outcomes
    .groupby(
        [
            "Item Code",
            "Item",
            "target_year",
        ],
        observed=True,
    )
    .agg(
        Eligible_observations=(
            "shortage_next_year",
            "size",
        ),
        Shortage_events=(
            "shortage_next_year",
            "sum",
        ),
        Countries_assessed=(
            "Area",
            "nunique",
        ),
    )
    .reset_index()
    .rename(
        columns={
            "target_year": "Outcome_year"
        }
    )
)

commodity_annual_history[
    "Event_rate"
] = (
    commodity_annual_history[
        "Shortage_events"
    ]
    / commodity_annual_history[
        "Eligible_observations"
    ]
)


# ------------------------------------------------------------
# 7. Create the continental evidence packet
# ------------------------------------------------------------

continental_packet = {
    "packet_type":
        "Continental analytical evidence",

    "interpretation": {
        "historical_results":
            "Observed outcomes in the analytical dataset",
        "forward_results": (
            "Model-generated screening estimates for 2024; "
            "not observed outcomes"
        ),
    },

    "historical_summary":
        dataframe_to_json_records(
            continental_historical_summary
        ),

    "annual_history":
        dataframe_to_json_records(
            continental_annual_history
        ),

    "country_evidence":
        dataframe_to_json_records(
            country_evidence_table.sort_values(
                "Historical_burden_rank"
            )
        ),

    "commodity_evidence":
        dataframe_to_json_records(
            commodity_evidence_table
        ),

    "high_warning_register_2024":
        dataframe_to_json_records(
            high_warning_register_2024
        ),

    "formal_warning_register_2024":
        dataframe_to_json_records(
            formal_warning_register_2024
        ),

    "rank_relationships":
        dataframe_to_json_records(
            country_rank_correlation
        ),

    "reporting_rules":
        reporting_framework[
            "mandatory_reporting_rules"
        ],

    "created_utc":
        datetime.now(timezone.utc).isoformat(),
}

with open(
    continental_packet_path,
    "w",
    encoding="utf-8",
) as packet_file:
    json.dump(
        continental_packet,
        packet_file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 8. Create one evidence packet for every country
# ------------------------------------------------------------

packet_index_records = []

country_names = sorted(
    country_evidence_table[
        "Area"
    ].unique()
)

for country_name in country_names:

    country_slug = make_safe_filename(
        country_name
    )

    country_packet_path = (
        country_packet_directory
        / f"country_{country_slug}_evidence.json"
    )

    country_summary = (
        country_evidence_table.loc[
            country_evidence_table[
                "Area"
            ].eq(country_name)
        ]
    )

    country_annual = (
        country_annual_history.loc[
            country_annual_history[
                "Area"
            ].eq(country_name)
        ]
        .sort_values("Outcome_year")
    )

    country_commodity = (
        country_commodity_history.loc[
            country_commodity_history[
                "Area"
            ].eq(country_name)
        ]
        .sort_values(
            [
                "Shortage_events",
                "Commodity_event_rate",
            ],
            ascending=[False, False],
        )
    )

    country_events = (
        historical_event_ledger.loc[
            historical_event_ledger[
                "Area"
            ].eq(country_name)
        ]
        .sort_values(
            [
                "event_year",
                "Item Code",
            ]
        )
    )

    country_forward = (
        canonical_risk_register_2024.loc[
            canonical_risk_register_2024[
                "Area"
            ].eq(country_name)
        ]
        .sort_values(
            "predicted_shortage_probability",
            ascending=False,
            na_position="last",
        )
    )

    country_trends = (
        report_analytical_panel.loc[
            report_analytical_panel[
                "Area"
            ].eq(country_name),
            available_trend_columns,
        ]
        .sort_values(
            ["Item Code", "Year"]
        )
    )

    country_packet = {
        "packet_type":
            "Country analytical evidence",

        "country":
            country_name,

        "country_summary":
            dataframe_first_record(
                country_summary
            ),

        "annual_observed_history":
            dataframe_to_json_records(
                country_annual
            ),

        "commodity_observed_history":
            dataframe_to_json_records(
                country_commodity
            ),

        "observed_shortage_events":
            dataframe_to_json_records(
                country_events
            ),

        "forward_2024_risk_register":
            dataframe_to_json_records(
                country_forward
            ),

        "analytical_trends_2010_2023":
            dataframe_to_json_records(
                country_trends
            ),

        "mandatory_interpretation": {
            "observed_events": (
                "Historical events recorded from known "
                "2011–2023 outcomes"
            ),
            "forward_warnings": (
                "Model-generated 2024 screening estimates, "
                "not observed events"
            ),
            "unscored": (
                "An unscored commodity must not be described "
                "as having zero risk"
            ),
            "causality": (
                "Supply and utilisation patterns may be "
                "described as associations or context, "
                "not proven causes"
            ),
        },

        "created_utc":
            datetime.now(timezone.utc).isoformat(),
    }

    with open(
        country_packet_path,
        "w",
        encoding="utf-8",
    ) as packet_file:
        json.dump(
            country_packet,
            packet_file,
            indent=2,
            ensure_ascii=False,
        )

    packet_index_records.append(
        {
            "Packet type":
                "Country",
            "Entity":
                country_name,
            "File":
                country_packet_path.name,
            "Path":
                str(country_packet_path),
            "Historical events":
                len(country_events),
            "Forward scored commodities":
                int(
                    country_forward[
                        "eligible_for_2024_scoring"
                    ].sum()
                ),
            "Forward formal warnings":
                int(
                    country_forward[
                        "predicted_shortage_warning"
                    ]
                    .fillna(0)
                    .sum()
                ),
        }
    )


# ------------------------------------------------------------
# 9. Create one evidence packet for every commodity
# ------------------------------------------------------------

commodity_identifiers = (
    commodity_evidence_table[
        ["Item Code", "Item"]
    ]
    .drop_duplicates()
    .sort_values("Item Code")
)

for commodity_record in (
    commodity_identifiers.to_dict(
        orient="records"
    )
):

    item_code = str(
        commodity_record["Item Code"]
    )

    item_name = str(
        commodity_record["Item"]
    )

    commodity_slug = make_safe_filename(
        item_name
    )

    commodity_packet_path = (
        commodity_packet_directory
        / (
            f"commodity_{item_code}_"
            f"{commodity_slug}_evidence.json"
        )
    )

    commodity_summary = (
        commodity_evidence_table.loc[
            commodity_evidence_table[
                "Item Code"
            ].astype(str).eq(item_code)
        ]
    )

    commodity_annual = (
        commodity_annual_history.loc[
            commodity_annual_history[
                "Item Code"
            ].astype(str).eq(item_code)
        ]
        .sort_values("Outcome_year")
    )

    commodity_country = (
        country_commodity_history.loc[
            country_commodity_history[
                "Item Code"
            ].astype(str).eq(item_code)
        ]
        .sort_values(
            [
                "Shortage_events",
                "Commodity_event_rate",
            ],
            ascending=[False, False],
        )
    )

    commodity_events = (
        historical_event_ledger.loc[
            historical_event_ledger[
                "Item Code"
            ].astype(str).eq(item_code)
        ]
        .sort_values(
            [
                "event_year",
                "Area",
            ]
        )
    )

    commodity_forward = (
        canonical_risk_register_2024.loc[
            canonical_risk_register_2024[
                "Item Code"
            ].astype(str).eq(item_code)
        ]
        .sort_values(
            "predicted_shortage_probability",
            ascending=False,
            na_position="last",
        )
    )

    commodity_trends = (
        report_analytical_panel.loc[
            report_analytical_panel[
                "Item Code"
            ].astype(str).eq(item_code),
            available_trend_columns,
        ]
        .sort_values(
            ["Area", "Year"]
        )
    )

    commodity_packet = {
        "packet_type":
            "Commodity analytical evidence",

        "item_code":
            item_code,

        "commodity":
            item_name,

        "commodity_summary":
            dataframe_first_record(
                commodity_summary
            ),

        "annual_observed_history":
            dataframe_to_json_records(
                commodity_annual
            ),

        "country_observed_history":
            dataframe_to_json_records(
                commodity_country
            ),

        "observed_shortage_events":
            dataframe_to_json_records(
                commodity_events
            ),

        "forward_2024_risk_register":
            dataframe_to_json_records(
                commodity_forward
            ),

        "analytical_trends_2010_2023":
            dataframe_to_json_records(
                commodity_trends
            ),

        "mandatory_interpretation": {
            "historical": (
                "Observed outcomes from 2011–2023"
            ),
            "forward": (
                "Model-generated 2024 screening estimates"
            ),
            "national_scope": (
                "Commodity warnings must not be generalised "
                "to national food insecurity"
            ),
        },

        "created_utc":
            datetime.now(timezone.utc).isoformat(),
    }

    with open(
        commodity_packet_path,
        "w",
        encoding="utf-8",
    ) as packet_file:
        json.dump(
            commodity_packet,
            packet_file,
            indent=2,
            ensure_ascii=False,
        )

    packet_index_records.append(
        {
            "Packet type":
                "Commodity",
            "Entity":
                item_name,
            "File":
                commodity_packet_path.name,
            "Path":
                str(commodity_packet_path),
            "Historical events":
                len(commodity_events),
            "Forward scored commodities":
                int(
                    commodity_forward[
                        "eligible_for_2024_scoring"
                    ].sum()
                ),
            "Forward formal warnings":
                int(
                    commodity_forward[
                        "predicted_shortage_warning"
                    ]
                    .fillna(0)
                    .sum()
                ),
        }
    )


# ------------------------------------------------------------
# 10. Save the reporting framework and packet index
# ------------------------------------------------------------

with open(
    reporting_framework_path,
    "w",
    encoding="utf-8",
) as framework_file:
    json.dump(
        reporting_framework,
        framework_file,
        indent=2,
        ensure_ascii=False,
    )

report_packet_index = pd.DataFrame(
    packet_index_records
)

report_packet_index.to_csv(
    packet_index_path,
    index=False,
)


# ------------------------------------------------------------
# 11. Validate the complete reporting input system
# ------------------------------------------------------------

country_packet_index = (
    report_packet_index.loc[
        report_packet_index[
            "Packet type"
        ].eq("Country")
    ]
)

commodity_packet_index = (
    report_packet_index.loc[
        report_packet_index[
            "Packet type"
        ].eq("Commodity")
    ]
)

assert len(country_packet_index) == 43
assert country_packet_index[
    "Entity"
].nunique() == 43

assert len(commodity_packet_index) == 8
assert commodity_packet_index[
    "Entity"
].nunique() == 8

assert country_packet_index[
    "Historical events"
].sum() == 334

assert commodity_packet_index[
    "Historical events"
].sum() == 334

assert country_packet_index[
    "Forward formal warnings"
].sum() == 77

assert commodity_packet_index[
    "Forward formal warnings"
].sum() == 77

assert report_packet_index[
    "Path"
].map(
    lambda path: Path(path).exists()
).all()

with open(
    continental_packet_path,
    "r",
    encoding="utf-8",
) as packet_file:
    reloaded_continental_packet = json.load(
        packet_file
    )

assert len(
    reloaded_continental_packet[
        "country_evidence"
    ]
) == 43

assert len(
    reloaded_continental_packet[
        "commodity_evidence"
    ]
) == 8


# ------------------------------------------------------------
# 12. Display the reporting-system summary
# ------------------------------------------------------------

reporting_system_summary = pd.DataFrame(
    {
        "Report input": [
            "Continental evidence packet",
            "Country evidence packets",
            "Commodity evidence packets",
            "Historical events represented in country packets",
            "Historical events represented in commodity packets",
            "Forward formal warnings represented",
        ],
        "Count": [
            1,
            len(country_packet_index),
            len(commodity_packet_index),
            int(
                country_packet_index[
                    "Historical events"
                ].sum()
            ),
            int(
                commodity_packet_index[
                    "Historical events"
                ].sum()
            ),
            int(
                country_packet_index[
                    "Forward formal warnings"
                ].sum()
            ),
        ],
    }
)

print("Analytical reporting-input system:")
display(reporting_system_summary)

print("\nCountry packet index:")
display(country_packet_index)

print("\nCommodity packet index:")
display(commodity_packet_index)

# Use the highest historical-burden country as the pilot.
pilot_country = (
    country_evidence_table
    .sort_values("Historical_burden_rank")
    .iloc[0]["Area"]
)

pilot_country_record = (
    country_packet_index.loc[
        country_packet_index[
            "Entity"
        ].eq(pilot_country)
    ]
    .iloc[0]
)

with open(
    pilot_country_record["Path"],
    "r",
    encoding="utf-8",
) as packet_file:
    pilot_country_packet = json.load(
        packet_file
    )

print(
    f"\nPilot country selected for template testing: "
    f"{pilot_country}"
)

print("\nPilot country headline evidence:")
display(
    pd.DataFrame(
        [
            pilot_country_packet[
                "country_summary"
            ]
        ]
    )
)

print(
    "\nThe analytical reporting framework and all "
    "52 evidence packets were created and validated.\n"
    "No narrative report has yet been generated."
)

Analytical reporting-input system:


,Report input,Count
0,Continental evidence packet,1
1,Country evidence packets,43
2,Commodity evidence packets,8
3,Historical events represented in country packets,334
4,Historical events represented in commodity pac...,334
5,Forward formal warnings represented,77



Country packet index:


,Packet type,Entity,File,Path,Historical events,Forward scored commodities,Forward formal warnings
0,Country,Algeria,country_algeria_evidence.json,/Users/adewale/Documents/food_security_predict...,0,3,1
1,Country,Angola,country_angola_evidence.json,/Users/adewale/Documents/food_security_predict...,3,6,1
2,Country,Botswana,country_botswana_evidence.json,/Users/adewale/Documents/food_security_predict...,8,5,1
3,Country,Burkina Faso,country_burkina_faso_evidence.json,/Users/adewale/Documents/food_security_predict...,9,6,1
4,Country,Cabo Verde,country_cabo_verde_evidence.json,/Users/adewale/Documents/food_security_predict...,7,5,0
5,Country,Cameroon,country_cameroon_evidence.json,/Users/adewale/Documents/food_security_predict...,3,8,0
6,Country,Comoros,country_comoros_evidence.json,/Users/adewale/Documents/food_security_predict...,2,6,2
7,Country,Congo,country_congo_evidence.json,/Users/adewale/Documents/food_security_predict...,5,6,2
8,Country,Côte d'Ivoire,country_cote_d_ivoire_evidence.json,/Users/adewale/Documents/food_security_predict...,3,8,1
9,Country,Democratic Republic of the Congo,country_democratic_republic_of_the_congo_evide...,/Users/adewale/Documents/food_security_predict...,5,5,2



Commodity packet index:


,Packet type,Entity,File,Path,Historical events,Forward scored commodities,Forward formal warnings
43,Commodity,Wheat and products,commodity_2511_wheat_and_products_evidence.json,/Users/adewale/Documents/food_security_predict...,57,43,16
44,Commodity,Maize and products,commodity_2514_maize_and_products_evidence.json,/Users/adewale/Documents/food_security_predict...,51,41,8
45,Commodity,Millet and products,commodity_2517_millet_and_products_evidence.json,/Users/adewale/Documents/food_security_predict...,34,21,7
46,Commodity,Sorghum and products,commodity_2518_sorghum_and_products_evidence.json,/Users/adewale/Documents/food_security_predict...,59,26,9
47,Commodity,Cassava and products,commodity_2532_cassava_and_products_evidence.json,/Users/adewale/Documents/food_security_predict...,24,28,6
48,Commodity,Yams,commodity_2535_yams_evidence.json,/Users/adewale/Documents/food_security_predict...,3,12,1
49,Commodity,Groundnuts,commodity_2552_groundnuts_evidence.json,/Users/adewale/Documents/food_security_predict...,64,35,20
50,Commodity,Rice and products,commodity_2807_rice_and_products_evidence.json,/Users/adewale/Documents/food_security_predict...,42,43,10



Pilot country selected for template testing: Uganda

Pilot country headline evidence:


,Area,Historical_eligible_observations,Historical_shortage_events,Historical_commodities_assessed,Historical_outcome_years_assessed,First_outcome_year,Last_outcome_year,Historical_event_rate,Historical_rate_lower_bound,Historical_rate_upper_bound,...,Forward_watch_share,Historical_burden_rank,Recent_burden_rank,Forward_screening_priority_rank,Historical_raw_rate_rank,Recent_raw_rate_rank,Forward_peak_risk_rank,Forward_warning_share_rank,Historical_rank_adjustment_for_denominator,Recent_rank_adjustment_for_denominator
0,Uganda,89,21,7,13,2011,2023,0.235955,0.159824,0.333936,...,1.0,1,1,3,1,1,10,4,0,0



The analytical reporting framework and all 52 evidence packets were created and validated.
No narrative report has yet been generated.


In [16]:
# ============================================================
# NOTEBOOK 5 — STEP 10
# PREPARE THE UGANDA PILOT REPORT EVIDENCE
#
# This creates the exact tables needed to write the first
# country report. It does not generate narrative conclusions.
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Load Uganda's packet sections
# ------------------------------------------------------------

pilot_country_name = "Uganda"

assert (
    pilot_country_packet["country"]
    == pilot_country_name
)

uganda_headline = pd.DataFrame(
    [
        pilot_country_packet[
            "country_summary"
        ]
    ]
)

uganda_annual_history = pd.DataFrame(
    pilot_country_packet[
        "annual_observed_history"
    ]
)

uganda_commodity_history = pd.DataFrame(
    pilot_country_packet[
        "commodity_observed_history"
    ]
)

uganda_observed_events = pd.DataFrame(
    pilot_country_packet[
        "observed_shortage_events"
    ]
)

uganda_forward_risks = pd.DataFrame(
    pilot_country_packet[
        "forward_2024_risk_register"
    ]
)

uganda_analytical_trends = pd.DataFrame(
    pilot_country_packet[
        "analytical_trends_2010_2023"
    ]
)


# ------------------------------------------------------------
# 2. Normalise key fields
# ------------------------------------------------------------

for dataframe in [
    uganda_commodity_history,
    uganda_observed_events,
    uganda_forward_risks,
    uganda_analytical_trends,
]:

    if (
        not dataframe.empty
        and "Item Code" in dataframe.columns
    ):
        dataframe["Item Code"] = (
            pd.to_numeric(
                dataframe["Item Code"],
                errors="raise",
            )
            .astype("int64")
            .astype(str)
        )

if not uganda_annual_history.empty:

    uganda_annual_history[
        "Outcome_year"
    ] = pd.to_numeric(
        uganda_annual_history[
            "Outcome_year"
        ],
        errors="raise",
    ).astype("int16")

if not uganda_observed_events.empty:

    uganda_observed_events[
        "event_year"
    ] = pd.to_numeric(
        uganda_observed_events[
            "event_year"
        ],
        errors="raise",
    ).astype("int16")

if not uganda_analytical_trends.empty:

    uganda_analytical_trends["Year"] = (
        pd.to_numeric(
            uganda_analytical_trends["Year"],
            errors="raise",
        )
        .astype("int16")
    )


# ------------------------------------------------------------
# 3. Create the headline summary
# ------------------------------------------------------------

headline_columns = [
    "Area",
    "Historical_eligible_observations",
    "Historical_shortage_events",
    "Historical_event_rate",
    "Historical_rate_lower_bound",
    "Historical_rate_upper_bound",
    "Historical_commodities_assessed",
    "Historical_burden_rank",
    "Recent_eligible_observations",
    "Recent_shortage_events",
    "Recent_event_rate",
    "Recent_rate_lower_bound",
    "Recent_rate_upper_bound",
    "Recent_burden_rank",
    "Commodities_with_shortages",
    "Commodities_with_recurrent_shortages",
    "Most_recent_shortage_year",
    "Forward_scored_commodities",
    "Forward_formal_warnings",
    "Forward_high_warnings",
    "Forward_average_probability",
    "Forward_maximum_probability",
    "Forward_screening_priority_rank",
    "Forward_peak_risk_rank",
]

available_headline_columns = [
    column
    for column in headline_columns
    if column in uganda_headline.columns
]

uganda_headline_view = (
    uganda_headline[
        available_headline_columns
    ]
)


# ------------------------------------------------------------
# 4. Summarise observed event years by commodity
# ------------------------------------------------------------

if uganda_observed_events.empty:

    uganda_event_year_summary = pd.DataFrame(
        columns=[
            "Item Code",
            "Item",
            "Observed_events",
            "Event_years",
            "First_event_year",
            "Most_recent_event_year",
        ]
    )

else:

    uganda_event_year_summary = (
        uganda_observed_events
        .groupby(
            ["Item Code", "Item"],
            observed=True,
        )
        .agg(
            Observed_events=(
                "event_year",
                "size",
            ),
            Event_years=(
                "event_year",
                lambda values: ", ".join(
                    map(
                        str,
                        sorted(
                            values.astype(int)
                        ),
                    )
                ),
            ),
            First_event_year=(
                "event_year",
                "min",
            ),
            Most_recent_event_year=(
                "event_year",
                "max",
            ),
        )
        .reset_index()
    )

uganda_commodity_event_view = (
    uganda_commodity_history
    .merge(
        uganda_event_year_summary,
        on=["Item Code", "Item"],
        how="left",
        validate="one_to_one",
    )
)

uganda_commodity_event_view[
    "Observed_events"
] = (
    uganda_commodity_event_view[
        "Observed_events"
    ]
    .fillna(0)
    .astype("int16")
)

uganda_commodity_event_view[
    "Event_years"
] = (
    uganda_commodity_event_view[
        "Event_years"
    ]
    .fillna("No observed shortage event")
)

uganda_commodity_event_view = (
    uganda_commodity_event_view
    .sort_values(
        [
            "Observed_events",
            "Commodity_event_rate",
        ],
        ascending=[False, False],
    )
)


# ------------------------------------------------------------
# 5. Prepare the complete 2024 risk view
# ------------------------------------------------------------

forward_columns = [
    "Item Code",
    "Item",
    "food_supply_kcal_cap_day",
    "eligible_for_2024_scoring",
    "scoring_exclusion_reason",
    "predicted_shortage_probability",
    "risk_band",
    "predicted_shortage_warning",
    "evidence_review_level",
    "eligible_for_headline_ranking",
    "missing_locked_feature_count",
    "non_year_features_outside_training_range_count",
    "pair_historical_rows",
    "pair_historical_shortages",
    "pair_historical_shortage_rate",
]

available_forward_columns = [
    column
    for column in forward_columns
    if column in uganda_forward_risks.columns
]

uganda_forward_risk_view = (
    uganda_forward_risks[
        available_forward_columns
    ]
    .sort_values(
        "predicted_shortage_probability",
        ascending=False,
        na_position="last",
    )
)


# ------------------------------------------------------------
# 6. Calculate recent descriptive context
# ------------------------------------------------------------
# These measures provide context only. They must not be called
# the causes of the model predictions.

context_metrics = [
    "food_supply_kcal_cap_day",
    "production_1000t",
    "import_quantity_1000t",
    "domestic_supply_1000t",
    "food_1000t",
]

available_context_metrics = [
    metric
    for metric in context_metrics
    if metric in uganda_analytical_trends.columns
]

trend_context_records = []

commodity_identifiers = (
    uganda_analytical_trends[
        ["Item Code", "Item"]
    ]
    .drop_duplicates()
)

for commodity_record in (
    commodity_identifiers.to_dict(
        orient="records"
    )
):

    item_code = commodity_record[
        "Item Code"
    ]

    item_name = commodity_record["Item"]

    commodity_trend = (
        uganda_analytical_trends.loc[
            uganda_analytical_trends[
                "Item Code"
            ].eq(item_code)
        ]
        .sort_values("Year")
        .set_index("Year")
    )

    record = {
        "Item Code": item_code,
        "Item": item_name,
    }

    for metric in available_context_metrics:

        value_2023 = (
            commodity_trend.at[2023, metric]
            if 2023 in commodity_trend.index
            else np.nan
        )

        value_2022 = (
            commodity_trend.at[2022, metric]
            if 2022 in commodity_trend.index
            else np.nan
        )

        recent_values = (
            commodity_trend.loc[
                commodity_trend.index
                .to_series()
                .between(2019, 2023)
                .to_numpy(),
                metric,
            ]
        )

        recent_mean = recent_values.mean(
            skipna=True
        )

        if (
            pd.notna(value_2023)
            and pd.notna(value_2022)
        ):
            absolute_change = (
                value_2023 - value_2022
            )
        else:
            absolute_change = np.nan

        if (
            pd.notna(value_2023)
            and pd.notna(value_2022)
            and value_2022 != 0
        ):
            percentage_change = (
                100
                * (value_2023 - value_2022)
                / abs(value_2022)
            )
        else:
            percentage_change = np.nan

        record[f"{metric}_2023"] = (
            value_2023
        )

        record[f"{metric}_2022"] = (
            value_2022
        )

        record[
            f"{metric}_change_2022_2023"
        ] = absolute_change

        record[
            f"{metric}_pct_change_2022_2023"
        ] = percentage_change

        record[
            f"{metric}_mean_2019_2023"
        ] = recent_mean

    trend_context_records.append(record)

uganda_recent_trend_context = pd.DataFrame(
    trend_context_records
)


# ------------------------------------------------------------
# 7. Join risk with selected recent context
# ------------------------------------------------------------

risk_context_columns = [
    "Item Code",
    "Item",
    "food_supply_kcal_cap_day_2023",
    "food_supply_kcal_cap_day_2022",
    "food_supply_kcal_cap_day_pct_change_2022_2023",
    "food_supply_kcal_cap_day_mean_2019_2023",
    "production_1000t_2023",
    "production_1000t_pct_change_2022_2023",
    "import_quantity_1000t_2023",
    "import_quantity_1000t_pct_change_2022_2023",
    "domestic_supply_1000t_2023",
    "domestic_supply_1000t_pct_change_2022_2023",
]

available_risk_context_columns = [
    column
    for column in risk_context_columns
    if column in uganda_recent_trend_context.columns
]

uganda_risk_context_view = (
    uganda_forward_risk_view
    .merge(
        uganda_recent_trend_context[
            available_risk_context_columns
        ],
        on=["Item Code", "Item"],
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 8. Integrity checks
# ------------------------------------------------------------

assert int(
    uganda_headline.iloc[0][
        "Historical_shortage_events"
    ]
) == len(uganda_observed_events)

assert len(uganda_observed_events) == 21

assert uganda_commodity_event_view[
    "Observed_events"
].sum() == 21

assert uganda_forward_risks[
    "eligible_for_2024_scoring"
].sum() == 7

assert (
    uganda_forward_risks[
        "predicted_shortage_warning"
    ]
    .fillna(0)
    .sum()
    == 5
)

assert (
    uganda_forward_risks[
        "predicted_shortage_probability"
    ]
    .ge(0.30)
    .sum()
    == 1
)


# ------------------------------------------------------------
# 9. Display the complete pilot evidence
# ------------------------------------------------------------

print("Uganda headline evidence:")
display(
    uganda_headline_view.style.format(
        {
            "Historical_event_rate": "{:.2%}",
            "Historical_rate_lower_bound": "{:.2%}",
            "Historical_rate_upper_bound": "{:.2%}",
            "Recent_event_rate": "{:.2%}",
            "Recent_rate_lower_bound": "{:.2%}",
            "Recent_rate_upper_bound": "{:.2%}",
            "Forward_average_probability": "{:.4f}",
            "Forward_maximum_probability": "{:.4f}",
        }
    )
)

print("\nUganda annual observed history:")
display(
    uganda_annual_history.style.format(
        {
            "Event_rate": "{:.2%}",
        }
    )
)

print(
    "\nUganda historical events by commodity:"
)
display(
    uganda_commodity_event_view.style.format(
        {
            "Commodity_event_rate": "{:.2%}",
        }
    )
)

print("\nUganda complete 2024 risk register:")
display(
    uganda_forward_risk_view.style.format(
        {
            "food_supply_kcal_cap_day": "{:.2f}",
            "predicted_shortage_probability":
                "{:.4f}",
            "pair_historical_shortage_rate":
                "{:.2%}",
        },
        na_rep="Not scored",
    )
)

print(
    "\nUganda recent supply context for interpreting "
    "the risk register:"
)
display(
    uganda_risk_context_view.style.format(
        {
            column: "{:.2f}"
            for column in (
                uganda_risk_context_view
                .select_dtypes(
                    include=[np.number]
                )
                .columns
            )
            if column not in [
                "predicted_shortage_warning",
                "missing_locked_feature_count",
                "non_year_features_outside_training_range_count",
                "pair_historical_rows",
                "pair_historical_shortages",
            ]
        },
        na_rep="Not recorded",
    )
)

print(
    "\nUganda pilot-report evidence is ready.\n"
    "No narrative interpretation has yet been added."
)

Uganda headline evidence:


,Area,Historical_eligible_observations,Historical_shortage_events,Historical_event_rate,Historical_rate_lower_bound,Historical_rate_upper_bound,Historical_commodities_assessed,Historical_burden_rank,Recent_eligible_observations,Recent_shortage_events,Recent_event_rate,Recent_rate_lower_bound,Recent_rate_upper_bound,Recent_burden_rank,Commodities_with_shortages,Commodities_with_recurrent_shortages,Most_recent_shortage_year,Forward_scored_commodities,Forward_formal_warnings,Forward_high_warnings,Forward_average_probability,Forward_maximum_probability,Forward_screening_priority_rank,Forward_peak_risk_rank
0,Uganda,89,21,23.60%,15.98%,33.39%,7,1,34,11,32.35%,19.13%,49.16%,1,7,6,2023.000000,7,5,1,0.1962,0.3182,3,10



Uganda annual observed history:


,Area,Outcome_year,Eligible_observations,Shortage_events,Commodities_assessed,Event_rate
0,Uganda,2011,7,0,7,0.00%
1,Uganda,2012,7,1,7,14.29%
2,Uganda,2013,7,2,7,28.57%
3,Uganda,2014,7,0,7,0.00%
4,Uganda,2015,7,1,7,14.29%
5,Uganda,2016,7,3,7,42.86%
6,Uganda,2017,7,2,7,28.57%
7,Uganda,2018,6,1,6,16.67%
8,Uganda,2019,7,5,7,71.43%
9,Uganda,2020,6,1,6,16.67%



Uganda historical events by commodity:


...


Uganda complete 2024 risk register:


,Item Code,Item,food_supply_kcal_cap_day,eligible_for_2024_scoring,scoring_exclusion_reason,predicted_shortage_probability,risk_band,predicted_shortage_warning,evidence_review_level,eligible_for_headline_ranking,missing_locked_feature_count,non_year_features_outside_training_range_count,pair_historical_rows,pair_historical_shortages,pair_historical_shortage_rate
0,2511,Wheat and products,106.69,True,Eligible for prospective scoring,0.3182,High warning: 30% or more,1.000000,Within usual historical support,True,2,0,13.000000,1.000000,7.69%
1,2807,Rice and products,90.70,True,Eligible for prospective scoring,0.2514,Warning: 13% to below 30%,1.000000,Within usual historical support,True,4,0,13.000000,3.000000,23.08%
2,2532,Cassava and products,123.62,True,Eligible for prospective scoring,0.2111,Warning: 13% to below 30%,1.000000,Within usual historical support,True,10,0,13.000000,3.000000,23.08%
3,2552,Groundnuts,23.54,True,Eligible for prospective scoring,0.1852,Warning: 13% to below 30%,1.000000,Within usual historical support,True,9,0,13.000000,4.000000,30.77%
4,2514,Maize and products,649.54,True,Eligible for prospective scoring,0.1822,Warning: 13% to below 30%,1.000000,Within usual historical support,True,3,0,13.000000,2.000000,15.38%
5,2517,Millet and products,16.32,True,Eligible for prospective scoring,0.1131,Watch: 5% to below 13%,0.000000,Within usual historical support,True,5,0,13.000000,3.000000,23.08%
6,2518,Sorghum and products,10.03,True,Eligible for prospective scoring,0.1125,Watch: 5% to below 13%,0.000000,Within usual historical support,True,4,0,11.000000,5.000000,45.45%
7,2535,Yams,Not scored,False,Current 2023 calorie availability is missing,Not scored,Not scored,Not scored,Not scored,False,76,3,0.000000,0.000000,Not scored



Uganda recent supply context for interpreting the risk register:


,Item Code,Item,food_supply_kcal_cap_day,eligible_for_2024_scoring,scoring_exclusion_reason,predicted_shortage_probability,risk_band,predicted_shortage_warning,evidence_review_level,eligible_for_headline_ranking,missing_locked_feature_count,non_year_features_outside_training_range_count,pair_historical_rows,pair_historical_shortages,pair_historical_shortage_rate,food_supply_kcal_cap_day_2023,food_supply_kcal_cap_day_2022,food_supply_kcal_cap_day_pct_change_2022_2023,food_supply_kcal_cap_day_mean_2019_2023,production_1000t_2023,production_1000t_pct_change_2022_2023,import_quantity_1000t_2023,import_quantity_1000t_pct_change_2022_2023,domestic_supply_1000t_2023,domestic_supply_1000t_pct_change_2022_2023
0,2511,Wheat and products,106.69,True,Eligible for prospective scoring,0.32,High warning: 30% or more,1.000000,Within usual historical support,True,2,0,13.000000,1.000000,0.08,106.69,82.07,30.00,95.93,25.00,0.00,839.00,21.07,775.00,30.91
1,2807,Rice and products,90.70,True,Eligible for prospective scoring,0.25,Warning: 13% to below 30%,1.000000,Within usual historical support,True,4,0,13.000000,3.000000,0.23,90.70,108.62,-16.50,97.00,365.00,5.19,368.00,-30.43,730.00,-16.09
2,2532,Cassava and products,123.62,True,Eligible for prospective scoring,0.21,Warning: 13% to below 30%,1.000000,Within usual historical support,True,10,0,13.000000,3.000000,0.23,123.62,159.37,-22.43,159.27,1877.00,-12.41,61.00,-73.25,1863.00,-21.19
3,2552,Groundnuts,23.54,True,Eligible for prospective scoring,0.19,Warning: 13% to below 30%,1.000000,Within usual historical support,True,9,0,13.000000,4.000000,0.31,23.54,50.60,-53.48,31.35,140.00,-48.53,119.00,128.85,253.00,-21.18
4,2514,Maize and products,649.54,True,Eligible for prospective scoring,0.18,Warning: 13% to below 30%,1.000000,Within usual historical support,True,3,0,13.000000,2.000000,0.15,649.54,695.72,-6.64,557.08,4945.00,4.37,38.00,-58.24,4359.00,-5.71
5,2517,Millet and products,16.32,True,Eligible for prospective scoring,0.11,Watch: 5% to below 13%,0.000000,Within usual historical support,True,5,0,13.000000,3.000000,0.23,16.32,18.94,-13.83,12.71,119.00,-7.75,11.00,-45.00,128.00,-13.51
6,2518,Sorghum and products,10.03,True,Eligible for prospective scoring,0.11,Watch: 5% to below 13%,0.000000,Within usual historical support,True,4,0,11.000000,5.000000,0.45,10.03,12.35,-18.79,8.89,266.00,-6.99,20.00,-25.93,220.00,-15.71
7,2535,Yams,Not recorded,False,Current 2023 calorie availability is missing,Not recorded,Not recorded,Not recorded,Not scored,False,76,3,0.000000,0.000000,Not recorded,Not recorded,Not recorded,Not recorded,0.00,Not recorded,Not recorded,Not recorded,Not recorded,0.00,Not recorded



Uganda pilot-report evidence is ready.
No narrative interpretation has yet been added.
